In [5]:
import json
import random

# ----------------------------
# CONFIG
# ----------------------------
N_SAMPLES = 6000   # total synthetic samples
TRAIN_OUT = "train_forecast_NEXT_big.jsonl"
VALID_OUT = "valid_forecast_NEXT_big.jsonl"
VALID_RATIO = 0.2

MACHINES = ["XP888A", "PQ667", "ZT901B", "LM442", "KJ3-A", "ALPHA01", "LINE-7"]

PREFIXES = [
    "", "tolong ", "bisa nggak ", "minta ", "tolong dong ",
    "plis ", "cek ", "coba ", "cuy ", "bos ", "bang ",
]

# noise informal / typo kecil
def noise_variants(s: str) -> str:
    ops = [
        lambda x: x,
        lambda x: x.lower(),
        lambda x: x.replace(" yang ", " yg ").replace("Yang ", "yg ").replace(" yang", " yg"),
        lambda x: x.replace(" menit", " mnt").replace("menit", "mnt"),
        lambda x: x.replace(" jam", " jm").replace("jam", "jm"),
        lambda x: x.replace(" kira kira", " kira2").replace("kira kira", "kira2").replace("kira-kira", "kira2"),
        lambda x: x.replace(" ke depan", " kedepan").replace(" ke depan", " kedepan"),
        lambda x: x.replace("apakah", "apa "),
    ]
    return random.choice(ops)(s)

HORIZON_STEPS = [3, 6, 12, 24, 30, 48]

ABS_WINDOWS = [
    ("22:00 hari ini - 06:00 besok", "ABSOLUTE_START", "HOUR"),
    ("jam 22:00 nanti s.d. jam 02:00 besok", "ABSOLUTE_START", "HOUR"),
    ("besok 08:00-12:00", "ABSOLUTE_START", "HOUR"),
    ("1-7 Jan 2026", "ABSOLUTE_START", "DAY"),
    ("10-14 minggu depan", "ABSOLUTE_START", "DAY"),
    ("sebelum jam 11 siang nanti", "ABSOLUTE_START", "HOUR"),
    ("sebelum jam 06:00 besok", "ABSOLUTE_START", "HOUR"),
]

TEMP_LIMITS = [60,65,70,75,80,85,90,95,100]
VIB_LIMITS = [15,18,20,22,25,30]
PRESSURE_LIMITS = [180,190,200,210,220,230]
AMP_LIMITS = [10,12,14,15,16,18]

# pools buat wording sinonim
FORECAST_VERB_POOL = [
    "prediksi", "ramalan", "ramalin", "forecast-in",
    "perkiraan", "estimasi", "estimate", "kira-kira",
    "project-kan", "proyeksikan", "coba lihat outlook",
    "kondisi nanti", "kondisi ke depan",
]

TREND_PHRASE_POOL = [
    "arah trennya gimana",
    "trennya naik/turun ga",
    "bakal stabil atau lonjak",
    "bakal normal atau spike",
    "ada indikasi naik tajam ga",
    "bakal anjlok apa aman",
    "masih aman gak",
]

ANOMALY_POOL = [
    "ada anomali", "ada anomali nggak",
    "ada gejala abnormal nggak",
    "ada perilaku nggak normal gak",
    "ada perilaku aneh gak",
    "ada yang ngaco ga",
    "keluar batas normal ga",
    "ada yang di luar normal ga",
    "ada spike aneh ga",
    "ada potensi overheat ga",
]

WHATIF_INTRO_POOL = [
    "kalau misalnya",
    "andaikan",
    "anggap aja",
    "anggap kondisi kita tuh",
    "anggap skenario nya gini:",
    "seandainya",
    "jika kita ubah datanya jadi",
]

WHATIF_ASK_POOL = [
    "kira-kira nanti jadi berapa?",
    "outputnya jadi berapa?",
    "berapa perubahan nilainya?",
    "berapa efeknya ke depan?",
    "gimana impactnya?",
    "apa efeknya 2 jam lagi?",
    "bakal naik jadi berapa?",
]

GENERAL_VALUE_FORMS = [
    "nilai mesin {m} {h} jam lagi berapa?",
    "berapa nilai mesin {m} {h} jam dari sekarang?",
    "{m} {h} jam lagi angkanya kira kira berapa?",
    "prediksi {m} {h} jam lagi jadi berapa?",
    "berapa kira2 value {m} {h} jam kedepan?",
    "berapa estimasi {m} {h} jam kedepan?",
    "berapa nanti bacaan {m} {h} jam lagi?",
]

ANOMALY_FORMS = [
    "coba prediksi semua mesin apakah akan ada anomali {h} jam ke depan?",
    "ada kemungkinan {anom} di semua mesin {h} jam kedepan?",
    "bakal ada perilaku {anom} di semua mesin dalam {h} jam kedepan ga?",
    "semua mesin aman ga {h} jam kedepan atau bakal {anom}?",
    "2 jam kedepan ada yg bakal ngaco ga di semua mesin?",
]

# =====================================================================
# Tambahan untuk skenario WHAT-IF
# =====================================================================

# referensi waktu lampau yang variatif:
PAST_REF_POOL = [
    # granularity menit
    "{ts} tadi (5 menit lalu)",
    "{ts} tadi (10 menit lalu)",
    "{ts} tadi (15 menit lalu)",
    "{ts} barusan (beberapa menit lalu)",

    # granularity jam
    "{ts} 1 jam lalu",
    "{ts} 2 jam lalu",
    "{ts} 3 jam lalu",

    # granularity hari
    "{ts} tadi pagi",
    "{ts} tadi siang",
    "{ts} tadi malam",
    "{ts} barusan malam ini",
    "{ts} kemarin",
    "{ts} kemarin siang",
    "{ts} kemarin malam",

    # granularity hari spesifik
    "hari ini jam {ts}",
    "siang ini jam {ts}",
    "pagi ini jam {ts}",
    "malam ini jam {ts}",
    "kemarin jam {ts}",
    "kemarin sore jam {ts}",
    "kemarin malam jam {ts}",
    "2 hari lalu jam {ts}",
    "3 hari lalu jam {ts}",

    # granularity minggu
    "awal minggu ini jam {ts}",
    "minggu lalu jam {ts}",
    "akhir minggu lalu jam {ts}",

    # granularity bulan
    "awal bulan ini jam {ts}",
    "bulan lalu jam {ts}",
    "awal bulan lalu jam {ts}",
    "akhir bulan lalu jam {ts}",
]

# bentuk kalimat what-if (pakai {past_ref} sekarang, bukan fix 'minggu lalu jam {ts}')
WHATIF_FORMS = [
    "{intro} nilai {past_ref} untuk {m} saya set jadi {val}, {ask}",
    "{intro} {m} di {past_ref} itu {val}, {ask}",
    "{intro} data {past_ref} diganti {val} untuk {m}, {ask}",
    "{intro} {past_ref} {m} = {val}, {ask}",
    "{intro} kalau catatan {m} {past_ref} dianggap {val}, {ask}",
    "{intro} anggap kondisi {m} {past_ref} nilainya {val}, {ask}",
    "{intro} misal {m} {past_ref} sebenarnya {val}, {ask}",
    "{intro} kita ubah histori {m} {past_ref} jadi {val}, {ask}",
    "{intro} andaikan waktu {past_ref} {m} = {val}, {ask}",
]


def choose_scope():
    mode = random.choice(["SINGLE","MULTI","ALL","SINGLE","SINGLE","ALL"])
    if mode == "SINGLE":
        machine = [random.choice(MACHINES)]
        return mode, machine
    if mode == "MULTI":
        m2 = random.sample(MACHINES, k=min(2, len(MACHINES)))
        return mode, m2
    return "ALL", []


def apply_prefix_and_noise(text: str) -> str:
    pref = random.choice(PREFIXES)
    noised = noise_variants(text)
    return pref + noised


# 1. FORECAST_VALUE
def make_forecast_value():
    scope, machines = choose_scope()
    gran = random.choice(["MINUTE","MINUTE","MINUTE","HOUR","DAY"])
    anchor_type = (
        random.choice(["NOW_START","ABSOLUTE_START"])
        if gran != "MINUTE"
        else random.choice(["NOW_START","NOW_START","ABSOLUTE_START"])
    )
    style = random.choice(["FIXED_STEPS","RANGE_FUTURE","CUSTOM_ABSOLUTE"])
    if anchor_type == "ABSOLUTE_START" and style == "RANGE_FUTURE":
        style = "CUSTOM_ABSOLUTE"

    verb = random.choice(FORECAST_VERB_POOL)

    if style == "FIXED_STEPS":
        steps = sorted(random.sample(HORIZON_STEPS, k=random.choice([1,2,3,4])))
        h_start = min(steps)
        h_end = max(steps)
        horizon_type = "FIXED_STEPS"
        time_complexity = "SINGLE_POINT" if len(steps)==1 else "RANGE_REL"

        if anchor_type == "ABSOLUTE_START":
            anchor_time_text = random.choice([
                "besok jam 09:00",
                "besok jam 10 pagi",
                "besok 08:00-12:00",
            ])
            horizon_type = "CUSTOM_ABSOLUTE"
            time_complexity = "SINGLE_POINT" if len(steps)==1 else "RANGE_ABS"
            h_start = None
            h_end = None
        else:
            anchor_time_text = "sekarang"

        if scope == "ALL":
            if len(steps)==1:
                base_q = f"{verb} semua mesin {h_end} menit dari sekarang"
            else:
                base_q = f"{verb} semua mesin di {', '.join(str(x) for x in steps)} menit berikutnya"
        elif scope == "MULTI":
            if len(steps)==1:
                base_q = f"{verb} {', '.join(machines)} untuk {h_end} menit lagi"
            else:
                base_q = f"{verb} {', '.join(machines)} buat {', '.join(str(x) for x in steps)} menit mendatang"
        else:
            # SINGLE
            if len(steps)==1:
                base_q = f"{verb} {machines[0]} {h_end} menit lagi"
            else:
                base_q = f"{verb} {machines[0]} di {', '.join(str(x) for x in steps)} menit lagi"

    elif style == "RANGE_FUTURE":
        start_m = random.choice([0,3,6])
        end_m = random.choice([12,24,30,48])
        horizon_type = "RANGE_FUTURE"
        time_complexity = "RANGE_REL"
        h_start = start_m
        h_end = end_m
        anchor_time_text = "sekarang"

        if scope == "ALL":
            base_q = f"{verb} kinerja semua mesin untuk {end_m} menit ke depan, interval per 3 menit"
        elif scope == "MULTI":
            base_q = f"{verb} outlook {', '.join(machines)} selama {end_m} menit depan"
        else:
            base_q = f"Saya mau {verb} {machines[0]} dari sekarang sampai {end_m} menit depan tiap 3 menit interval"

    else:
        # CUSTOM_ABSOLUTE
        abs_win, abs_anchor_type, abs_gran = random.choice(ABS_WINDOWS)
        anchor_type = abs_anchor_type
        gran = abs_gran
        horizon_type = "CUSTOM_ABSOLUTE"
        time_complexity = random.choice(["RANGE_ABS","SINGLE_POINT"])
        h_start = None
        h_end = None
        anchor_time_text = abs_win

        if scope == "ALL":
            base_q = f"{verb} semua mesin untuk shift ini ({abs_win})"
        elif scope == "MULTI":
            base_q = f"{verb} {', '.join(machines)} di window {abs_win}"
        else:
            base_q = f"{verb} {machines[0]} pada {abs_win}"

    text = apply_prefix_and_noise(base_q)

    labels = {
        "intent": "FORECAST_VALUE",
        "scope": scope,
        "machines": machines,
        "time_granularity": gran,
        "anchor_time_type": anchor_type,
        "time_complexity": time_complexity,
        "horizon_type": horizon_type,
        "horizon_start_min": h_start,
        "horizon_end_min": h_end,
        "anchor_time_text": anchor_time_text,
        "threshold_target": {
            "metric": None,
            "operator": None,
            "value": None,
        },
        "what_if_context": None,
    }
    return {"text": text, "labels": labels}


# 2. FORECAST_TREND (arah, aman/nggak, anomali/anomali mirip)
def make_forecast_trend():
    scope, machines = choose_scope()

    gran = random.choice(["MINUTE","MINUTE","HOUR","DAY"])
    anchor_type = (
        random.choice(["NOW_START","ABSOLUTE_START"])
        if gran != "MINUTE"
        else random.choice(["NOW_START","NOW_START","ABSOLUTE_START"])
    )
    style = random.choice(["FIXED_STEPS","RANGE_FUTURE","CUSTOM_ABSOLUTE"])
    if anchor_type == "ABSOLUTE_START" and style == "RANGE_FUTURE":
        style = "CUSTOM_ABSOLUTE"

    trend_phrase = random.choice(TREND_PHRASE_POOL)

    if style == "FIXED_STEPS":
        step = random.choice(HORIZON_STEPS)
        horizon_type = "FIXED_STEPS"
        time_complexity = "SINGLE_POINT"
        h_start = step
        h_end = step

        if anchor_type == "ABSOLUTE_START":
            anchor_time_text = random.choice([
                "jam 10 pagi nanti",
                "besok jam 09:00",
                "shift malam nanti",
            ])
            horizon_type = "CUSTOM_ABSOLUTE"
            time_complexity = "SINGLE_POINT"
            h_start = None
            h_end = None
        else:
            anchor_time_text = "sekarang"

        if scope == "ALL":
            base_q = f"Semua mesin {step} menit ke depan {trend_phrase}?"
        elif scope == "MULTI":
            base_q = f"{', '.join(machines)} {step} menit lagi {trend_phrase}? Mana yg risk duluan?"
        else:
            base_q = f"{machines[0]} {step} menit lagi {trend_phrase}?"

    elif style == "RANGE_FUTURE":
        start_m = random.choice([0,3,6])
        end_m = random.choice([12,24,30,48])
        horizon_type = "RANGE_FUTURE"
        time_complexity = "RANGE_REL"
        h_start = start_m
        h_end = end_m
        anchor_time_text = "sekarang"

        anomaly_term = random.choice(ANOMALY_POOL)

        if scope == "ALL":
            base_q = (
                f"Dari sekarang sampai {end_m} menit lagi, mesin mana paling agresif naiknya? "
                f"Ada potensi {anomaly_term}?"
            )
        elif scope == "MULTI":
            base_q = (
                f"Tolong bandingin {', '.join(machines)} dari menit {start_m} "
                f"sampai {end_m}, mana yg paling cepat naik? Ada {anomaly_term}?"
            )
        else:
            base_q = (
                f"Gue mau liat {machines[0]} menit {start_m} sampe {end_m} "
                f"bakal stabil atau lonjak? Ada tanda {anomaly_term}?"
            )

    else:
        # CUSTOM_ABSOLUTE
        abs_win, abs_anchor_type, abs_gran = random.choice(ABS_WINDOWS)
        anchor_type = abs_anchor_type
        gran = abs_gran
        horizon_type = "CUSTOM_ABSOLUTE"
        time_complexity = random.choice(["RANGE_ABS","SINGLE_POINT"])
        h_start = None
        h_end = None
        anchor_time_text = abs_win

        anomaly_term = random.choice(ANOMALY_POOL)

        if scope == "ALL":
            base_q = f"Prediksi semua mesin pada {abs_win} aman gak atau bakal {anomaly_term}?"
        elif scope == "MULTI":
            base_q = f"Selama {abs_win}, {', '.join(machines)} aman ga? ada {anomaly_term} ga?"
        else:
            base_q = f"{machines[0]} selama {abs_win} masih aman gak atau bakal {anomaly_term}?"

        # 30% special form "anomali X jam ke depan"
        if random.random() < 0.3:
            hrs = random.choice([1,2,3,4])
            mins_total = hrs * 60
            base_q = f"Coba prediksi semua mesin apakah akan ada {random.choice(ANOMALY_POOL)} {hrs} jam ke depan?"
            anchor_type = "NOW_START"
            time_complexity = "RANGE_REL"
            horizon_type = "RANGE_FUTURE"
            h_start = 0
            h_end = mins_total
            anchor_time_text = "sekarang"

    text = apply_prefix_and_noise(base_q)

    labels = {
        "intent": "FORECAST_TREND",
        "scope": scope,
        "machines": machines,
        "time_granularity": gran,
        "anchor_time_type": anchor_type,
        "time_complexity": time_complexity,
        "horizon_type": horizon_type,
        "horizon_start_min": h_start,
        "horizon_end_min": h_end,
        "anchor_time_text": anchor_time_text,
        "threshold_target": {
            "metric": None,
            "operator": None,
            "value": None,
        },
        "what_if_context": None,
    }
    return {"text": text, "labels": labels}


# 3. THRESHOLD_TIME
def make_threshold_time():
    scope, machines = choose_scope()

    metric_type = random.choice(["temperature","vibration","pressure","ampere"])
    if metric_type == "temperature":
        val = random.choice(TEMP_LIMITS)
        oper = random.choice(["RISE_ABOVE","DROP_BELOW"])
        cond_phrase = (
            f"naik di atas {val} derajat" if oper=="RISE_ABOVE"
            else f"turun di bawah {val} derajat"
        )
    elif metric_type == "vibration":
        val = random.choice(VIB_LIMITS)
        oper = random.choice(["RISE_ABOVE","DROP_BELOW"])
        cond_phrase = (
            f"lewat batas getaran {val}" if oper=="RISE_ABOVE"
            else f"balik normal di bawah {val} getaran"
        )
    elif metric_type == "pressure":
        val = random.choice(PRESSURE_LIMITS)
        oper = "RISE_ABOVE"
        cond_phrase = f"melewati tekanan {val} bar"
    else:
        val = random.choice(AMP_LIMITS)
        oper = random.choice(["RISE_ABOVE","DROP_BELOW"])
        cond_phrase = (
            f"lonjakan arus di atas {val} Ampere" if oper=="RISE_ABOVE"
            else f"turun lagi di bawah {val} Ampere"
        )

    gran = random.choice(["MINUTE","MINUTE","MINUTE","HOUR"])
    anchor_type = random.choice(["NOW_START","ABSOLUTE_START"])

    if anchor_type == "NOW_START":
        anchor_txt = "sekarang"
        if random.random() < 0.7:
            end_m = random.choice([20,24,30,48,120])
            time_complexity = "RANGE_REL"
            horizon_type = "UNTIL_THRESHOLD"
            h_start = 0
            h_end = end_m
            time_phrase = f"dalam {end_m} menit ke depan"
        else:
            time_complexity = "RANGE_REL"
            horizon_type = "UNTIL_THRESHOLD"
            h_start = 0
            h_end = None
            time_phrase = "kalau trendnya lanjut kayak sekarang"
    else:
        abs_win, abs_anchor_type, abs_gran = random.choice(ABS_WINDOWS)
        anchor_type = abs_anchor_type
        gran = abs_gran
        anchor_txt = abs_win
        time_complexity = "RANGE_ABS"
        horizon_type = "CUSTOM_ABSOLUTE"
        h_start = None
        h_end = None
        time_phrase = abs_win

    # variasi wording yang mirip
    if scope == "ALL":
        base_q_opts = [
            f"Kapan kira-kira semua mesin bakal {cond_phrase}? ({time_phrase})",
            f"Perkiraan jam berapa semua mesin mulai {cond_phrase}? {time_phrase}",
            f"Semua mesin bakal {cond_phrase} kapan? {time_phrase}",
            f"Kasih estimasi kapan semua mesin {cond_phrase}, {time_phrase}",
        ]
        base_q = random.choice(base_q_opts)
    elif scope == "MULTI":
        names = ", ".join(machines)
        base_q_opts = [
            f"Buat {names}, {time_phrase} bakal {cond_phrase} gak? jam brp kira2 pertama?",
            f"{names} kira2 kapan mulai {cond_phrase}? {time_phrase}",
            f"{names} bakal {cond_phrase} kapan pertama kalinya ({time_phrase})?",
        ]
        base_q = random.choice(base_q_opts)
    else:
        m = machines[0]
        base_q_opts = [
            f"{m} bakal {cond_phrase} kapan? {time_phrase}",
            f"Kapan {m} mulai {cond_phrase}? {time_phrase}",
            f"Jam berapa pertama kali {m} {cond_phrase} ({time_phrase})?",
            f"Perkiraan menit berapa {m} {cond_phrase}? {time_phrase}",
        ]
        base_q = random.choice(base_q_opts)

    text = apply_prefix_and_noise(base_q)

    labels = {
        "intent": "THRESHOLD_TIME",
        "scope": scope,
        "machines": machines,
        "time_granularity": gran,
        "anchor_time_type": anchor_type,
        "time_complexity": time_complexity,
        "horizon_type": horizon_type,
        "horizon_start_min": h_start,
        "horizon_end_min": h_end,
        "anchor_time_text": anchor_txt,
        "threshold_target": {
            "metric": metric_type,
            "operator": oper,
            "value": val,
        },
        "what_if_context": None,
    }
    return {"text": text, "labels": labels}


# 4. GENERAL_VALUE_STYLE → pertanyaan super langsung per jam
def make_general_value_style():
    scope = "SINGLE"
    m = random.choice(MACHINES)
    machines = [m]

    hrs = random.choice([1,2,3,4,6,8])
    mins_ahead = hrs * 60

    template = random.choice(GENERAL_VALUE_FORMS)
    base_q = template.format(m=m, h=hrs)

    text = apply_prefix_and_noise(base_q)

    labels = {
        "intent": "FORECAST_VALUE",
        "scope": scope,
        "machines": machines,
        "time_granularity": "HOUR",
        "anchor_time_type": "NOW_START",
        "time_complexity": "SINGLE_POINT",
        "horizon_type": "FIXED_STEPS",
        "horizon_start_min": mins_ahead,
        "horizon_end_min": mins_ahead,
        "anchor_time_text": "sekarang",
        "threshold_target": {
            "metric": None,
            "operator": None,
            "value": None,
        },
        "what_if_context": None,
    }
    return {"text": text, "labels": labels}


# 5. ANOMALY_CHECK_EXPLICIT → pakai kata anomali / abnormal variasi mirip
def make_anomaly_check_style():
    scope = "ALL"
    machines = []

    hrs = random.choice([1,2,3,4])
    mins_total = hrs * 60

    anomaly_term = random.choice(ANOMALY_POOL)
    template = random.choice(ANOMALY_FORMS)
    # di template ada {h} jam kedepan dan {anom}
    base_q = template.format(h=hrs, anom=anomaly_term)

    text = apply_prefix_and_noise(base_q)

    labels = {
        "intent": "FORECAST_TREND",
        "scope": scope,
        "machines": machines,
        "time_granularity": "HOUR",
        "anchor_time_type": "NOW_START",
        "time_complexity": "RANGE_REL",
        "horizon_type": "RANGE_FUTURE",
        "horizon_start_min": 0,
        "horizon_end_min": mins_total,
        "anchor_time_text": "sekarang",
        "threshold_target": {
            "metric": None,
            "operator": None,
            "value": None,
        },
        "what_if_context": None,
    }
    return {"text": text, "labels": labels}


# 6. WHAT_IF_SCENARIO → simulasi kondisi di masa lalu diubah
def make_what_if_scenario():
    scope = "SINGLE"
    m = random.choice(MACHINES)
    machines = [m]

    # jam spesifik untuk disisipkan ke past_ref
    ts = random.choice(["08:00", "09:30", "10:00", "13:15", "14:00", "18:30", "22:00"])

    # pilih referensi waktu lampau dan render {ts} ke dalamnya
    raw_past_ref = random.choice(PAST_REF_POOL)
    past_ref_txt = raw_past_ref.format(ts=ts)

    # nilai baru hipotetis
    new_val = random.choice([60,65,70,75,80,90,95,100,110,120,150])
    # horizon masa depan yang mau ditanya impactnya
    hrs_ahead = random.choice([1,2,3,4,6])
    mins_ahead = hrs_ahead * 60

    intro = random.choice(WHATIF_INTRO_POOL)
    ask = random.choice(WHATIF_ASK_POOL)
    template = random.choice(WHATIF_FORMS)

    # build kalimat utama
    base_q = template.format(
        intro=intro,
        past_ref=past_ref_txt,
        m=m,
        val=new_val,
        ask=ask
    )

    # kadang kita tambahin bentuk lain yang pakai kosakata beda tapi sama maksud
    # (supaya nambah variasi "efeknya apa nanti")
    if random.random() < 0.3:
        alt = (
            f"{intro} kalau histori {m} di {past_ref_txt} gue ubah jadi {new_val}, "
            f"{random.choice(WHATIF_ASK_POOL)}"
        )
        base_q = random.choice([base_q, alt])

    # apply prefix (tolong / bisa nggak) dan noise (kira2 -> kira2, menit->mnt, dll)
    text = apply_prefix_and_noise(base_q)

    # simpan konteks what-if secara eksplisit ke label
    # ini penting buat downstream supaya tau "oh ini simulasi, bukan kondisi real"
    what_if_desc = f"override nilai historical {m} @ {past_ref_txt} = {new_val}"

    labels = {
        "intent": "WHAT_IF_FORECAST",          # intent baru tetap sama
        "scope": scope,
        "machines": machines,
        "time_granularity": "HOUR",            # karena kita ngomong hrs_ahead
        "anchor_time_type": "NOW_START",       # efek di masa depan relatif dari sekarang
        "time_complexity": "SINGLE_POINT",
        "horizon_type": "FIXED_STEPS",
        "horizon_start_min": mins_ahead,
        "horizon_end_min": mins_ahead,
        "anchor_time_text": "sekarang",
        "threshold_target": {
            "metric": None,
            "operator": None,
            "value": None,
        },
        "what_if_context": what_if_desc,
    }
    return {"text": text, "labels": labels}


def generate_sample():
    # weight jenis-jenis query
    typ = random.choices(
        [
            "FORECAST_VALUE",
            "FORECAST_TREND",
            "THRESHOLD_TIME",
            "GENERAL_VALUE",
            "ANOMALY_CHECK",
            "WHAT_IF",
        ],
        weights=[0.27, 0.20, 0.18, 0.12, 0.12, 0.11],
        k=1
    )[0]

    if typ == "FORECAST_VALUE":
        return make_forecast_value()
    elif typ == "FORECAST_TREND":
        return make_forecast_trend()
    elif typ == "THRESHOLD_TIME":
        return make_threshold_time()
    elif typ == "GENERAL_VALUE":
        return make_general_value_style()
    elif typ == "ANOMALY_CHECK":
        return make_anomaly_check_style()
    else:
        return make_what_if_scenario()


# MAIN BUILD
samples = [generate_sample() for _ in range(N_SAMPLES)]
random.shuffle(samples)

split_idx = int(len(samples) * (1 - VALID_RATIO))
train_data = samples[:split_idx]
valid_data = samples[split_idx:]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(TRAIN_OUT, train_data)
write_jsonl(VALID_OUT, valid_data)

print("DONE")
print("train size:", len(train_data))
print("valid size:", len(valid_data))
print("example:\n", json.dumps(samples[0], ensure_ascii=False, indent=2))


DONE
train size: 4800
valid size: 1200
example:
 {
  "text": "plis anggap aja misal pq667 siang ini jam 09:30 sebenarnya 150, gimana impactnya?",
  "labels": {
    "intent": "WHAT_IF_FORECAST",
    "scope": "SINGLE",
    "machines": [
      "PQ667"
    ],
    "time_granularity": "HOUR",
    "anchor_time_type": "NOW_START",
    "time_complexity": "SINGLE_POINT",
    "horizon_type": "FIXED_STEPS",
    "horizon_start_min": 60,
    "horizon_end_min": 60,
    "anchor_time_text": "sekarang",
    "threshold_target": {
      "metric": null,
      "operator": null,
      "value": null
    },
    "what_if_context": "override nilai historical PQ667 @ siang ini jam 09:30 = 150"
  }
}


In [10]:
import json, os

# lokasi raw dataset besar hasil generator synthetic
RAW_TRAIN = "/workspace/next/train_forecast_NEXT_big.jsonl"
RAW_VALID = "/workspace/next/valid_forecast_NEXT_big.jsonl"

OUT_DIR   = "/workspace/next"  # sama kayak DATA_DIR di trainer

os.makedirs(OUT_DIR, exist_ok=True)

def load_jsonl_all(p):
    rows=[]
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

full_rows = load_jsonl_all(RAW_TRAIN) + load_jsonl_all(RAW_VALID)

# helper safe get
def safe_num(v, fallback=-1):
    return fallback if v is None else v

intent_rows              = []
scope_rows               = []
timecombo_rows           = []
whatif_flag_rows         = []
threshold_metric_rows    = []
threshold_operator_rows  = []

horizon_rows             = []
threshold_value_rows     = []

for r in full_rows:
    qtext = r["text"]
    lab   = r["labels"]

    intent            = lab["intent"]
    scope             = lab["scope"]
    time_granularity  = lab["time_granularity"]
    anchor_time_type  = lab["anchor_time_type"]
    time_complexity   = lab["time_complexity"]
    horizon_type      = lab["horizon_type"]
    h_start           = safe_num(lab["horizon_start_min"], -1)
    h_end             = safe_num(lab["horizon_end_min"],   -1)
    thr               = lab["threshold_target"]
    whatif_context    = lab["what_if_context"]

    # ---------- intent_head ----------
    intent_rows.append({
        "text": qtext,
        "label": intent
    })

    # ---------- scope_head ----------
    scope_rows.append({
        "text": qtext,
        "label": scope
    })

    # ---------- timecombo_head ----------
    combo_label = f"{time_granularity}|{anchor_time_type}|{time_complexity}|{horizon_type}"
    timecombo_rows.append({
        "text": qtext,
        "label": combo_label
    })

    # ---------- whatif_flag_head ----------
    flag = "IS_WHAT_IF" if (intent == "WHAT_IF_FORECAST" or (whatif_context is not None)) else "NORMAL"
    whatif_flag_rows.append({
        "text": qtext,
        "label": flag
    })

    # ---------- horizon_head (regression) ----------
    # NOTE: even kalau RANGE_ABS (pakai shift malam dll) -> boleh aja -1, -1.
    horizon_rows.append({
        "text": qtext,
        "y_start": h_start,
        "y_end": h_end
    })

    # ---------- threshold heads ----------
    # metric/operator/value ONLY meaningful kalau intent == THRESHOLD_TIME
    # else -> NONE / -1
    metric_label = "NONE"
    oper_label   = "NONE"
    thr_value    = -1

    if thr is not None:
        metric_label = thr.get("metric", None)
        oper_label   = thr.get("operator", None)
        tval         = thr.get("value", None)

        if intent != "THRESHOLD_TIME":
            metric_label = "NONE"
            oper_label   = "NONE"
            thr_value    = -1
        else:
            # we're in THRESHOLD_TIME
            if metric_label is None: metric_label = "NONE"
            if oper_label   is None: oper_label   = "NONE"
            thr_value = -1 if tval is None else float(tval)

    threshold_metric_rows.append({
        "text": qtext,
        "label": metric_label
    })
    threshold_operator_rows.append({
        "text": qtext,
        "label": oper_label
    })
    threshold_value_rows.append({
        "text": qtext,
        "y_val": thr_value
    })

def dump_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for rr in rows:
            f.write(json.dumps(rr, ensure_ascii=False) + "\n")

dump_jsonl(os.path.join(OUT_DIR, "train_intent.jsonl"),              intent_rows)
dump_jsonl(os.path.join(OUT_DIR, "train_scope.jsonl"),               scope_rows)
dump_jsonl(os.path.join(OUT_DIR, "train_timecombo.jsonl"),           timecombo_rows)
dump_jsonl(os.path.join(OUT_DIR, "train_whatif_flag.jsonl"),         whatif_flag_rows)

dump_jsonl(os.path.join(OUT_DIR, "train_threshold_metric.jsonl"),    threshold_metric_rows)
dump_jsonl(os.path.join(OUT_DIR, "train_threshold_operator.jsonl"),  threshold_operator_rows)

dump_jsonl(os.path.join(OUT_DIR, "train_horizon.jsonl"),             horizon_rows)
dump_jsonl(os.path.join(OUT_DIR, "train_threshold_value.jsonl"),     threshold_value_rows)

print("done building per-head jsonl into", OUT_DIR)
print("example intent row:", intent_rows[0])
print("example horizon row:", horizon_rows[0])


done building per-head jsonl into /workspace/next
example intent row: {'text': 'plis anggap aja misal pq667 siang ini jam 09:30 sebenarnya 150, gimana impactnya?', 'label': 'WHAT_IF_FORECAST'}
example horizon row: {'text': 'plis anggap aja misal pq667 siang ini jam 09:30 sebenarnya 150, gimana impactnya?', 'y_start': 60, 'y_end': 60}


In [ ]:
# =============================================================
# Forecast — Multi Head Trainer
# (intent • scope • timecombo • horizon • threshold • whatif)
# =============================================================

# ============ (0) CONFIG ============
DATA_DIR   = "/workspace/next"   # ganti sesuai folder dataset hasil generator tadi
OUT_ROOT   = "./runs_forecast_heads_v1"
BASE_MODEL = "indobenchmark/indobert-base-p1"     # atau "roberta-base" kalau mau english-ish

EPOCHS  = 20
LR      = 3e-5
BATCH   = 16
MAX_LEN = 128
SEED    = 42

import os, json, random, numpy as np, warnings
from pathlib import Path
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, mean_squared_error
from sklearn.model_selection import StratifiedShuffleSplit

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_ROOT, exist_ok=True)

# -------------------------------------------------
# Helper: load jsonl (generic)
# -------------------------------------------------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

# -------------------------------------------------
# 1) MULTICLASS TRAINER
# -------------------------------------------------
def compute_metrics_mc(p):
    logits, y = p
    preds = logits.argmax(axis=1)
    return {
        "acc": accuracy_score(y, preds),
        "f1_macro": f1_score(y, preds, average="macro")
    }

def stratified_split(rows, label_key="label", test_size=0.12, seed=42):
    labels = [r[label_key] for r in rows]
    idx = np.arange(len(rows))
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, va_idx = next(sss.split(idx, labels))
    train_rows = [rows[i] for i in tr_idx]
    valid_rows = [rows[i] for i in va_idx]
    return train_rows, valid_rows

def train_multiclass_head(model_name, train_jsonl, out_dir,
                          label_key="label", epochs=6, lr=3e-5,
                          batch=16, max_len=128, seed=42):
    """
    train_jsonl rows must have: {"text": "...", "label": "..."}
    We'll auto-split valid stratified 12%.
    """
    rows = load_jsonl(train_jsonl)
    train_rows, valid_rows = stratified_split(rows, label_key=label_key, test_size=0.12, seed=seed)

    # build huggingface DatasetDict
    train_ds = Dataset.from_list(train_rows)
    valid_ds = Dataset.from_list(valid_rows)
    ds = DatasetDict({"train": train_ds, "validation": valid_ds})

    # build label maps
    labels_sorted = sorted(list({ex[label_key] for ex in train_rows}))
    label2id = {l:i for i,l in enumerate(labels_sorted)}
    id2label = {i:l for l,i in label2id.items()}

    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(batch):
        enc = tok(batch["text"], truncation=True, padding="max_length", max_length=max_len)
        enc["labels"] = [label2id[y] for y in batch[label_key]]
        return enc

    enc = ds.map(_pp, batched=True, remove_columns=["text",label_key])
    cols = ["input_ids","attention_mask","labels"]
    if "token_type_ids" in enc["train"].features:
        cols.append("token_type_ids")
    enc.set_format(type="torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels_sorted),
        id2label=id2label,
        label2id=label2id
    )

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        metric_for_best_model="eval_f1_macro",
        save_total_limit=2,
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc["train"],
        eval_dataset=enc["validation"],
        tokenizer=tok,
        compute_metrics=compute_metrics_mc,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=20)]
    )

    trainer.train()
    eval_res = trainer.evaluate()

    # save final model
    final_dir = os.path.join(out_dir, "final_model")
    os.makedirs(final_dir, exist_ok=True)
    trainer.save_model(final_dir)
    tok.save_pretrained(final_dir)

    # classification report
    pred_out = trainer.predict(enc["validation"])
    preds = np.argmax(pred_out.predictions, axis=1)
    y_true = pred_out.label_ids
    rep = classification_report(
        y_true,
        preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        digits=4
    )
    with open(os.path.join(out_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)

    # label maps
    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump(
            {"id2label": id2label, "label2id": {v:k for k,v in id2label.items()}},
            f, ensure_ascii=False, indent=2
        )

    return final_dir, eval_res, id2label


# -------------------------------------------------
# 2) REGRESSION TRAINER (horizon_start_min / horizon_end_min, threshold_value)
# -------------------------------------------------

from torch import nn

class BertRegressor(nn.Module):
    """
    Simple regression head on top of base transformer.
    We'll reuse pretrained model's encoder via AutoModelForSequenceClassification's config trick:
    Actually easier: load AutoModelForSequenceClassification then replace classifier.
    BUT we'll just build from AutoModelForSequenceClassification and set num_labels = out_dim,
    and use MSE loss manually in Trainer via custom compute_loss.
    """
    def __init__(self, base_name, out_dim=1):
        super().__init__()
        from transformers import AutoModel
        self.backbone = AutoModel.from_pretrained(base_name)
        hidden = self.backbone.config.hidden_size
        self.reg_head = nn.Linear(hidden, out_dim)
        self.config = self.backbone.config
        self.out_dim = out_dim

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # use CLS token
        cls = out.last_hidden_state[:,0,:]
        preds = self.reg_head(cls)
        loss = None
        if labels is not None:
            loss_fn = nn.MSELoss()
            loss = loss_fn(preds, labels.float())
        return {"loss": loss, "logits": preds}


def make_regression_dataset(rows, y_keys):
    """
    rows: [{"text": "...", "y_start": 120, "y_end": 240}, ...]
    y_keys: list[str] of target names in each row -> we concat into vector
    returns DatasetDict with fields: text, targets(list[float])
    """
    X = []
    Y = []
    for r in rows:
        X.append(r["text"])
        Y.append([ float(r[k]) for k in y_keys ])
    return X, Y

def split_simple(rows, test_size=0.12, seed=42):
    # not stratified here (regression)
    idx = np.arange(len(rows))
    np.random.RandomState(seed).shuffle(idx)
    cut = int(len(rows)*(1-test_size))
    tr_idx, va_idx = idx[:cut], idx[cut:]
    train_rows = [rows[i] for i in tr_idx]
    valid_rows = [rows[i] for i in va_idx]
    return train_rows, valid_rows

def train_regression_head(model_name, train_jsonl, out_dir,
                          y_keys, epochs=6, lr=3e-5, batch=16,
                          max_len=128, seed=42):
    """
    train_jsonl rows must have: {"text": "...", "y_start": ..., "y_end": ...}
    We'll auto split.
    y_keys = ["y_start","y_end"]  (multi-dim)  OR ["y_val"] (single)
    """
    rows = load_jsonl(train_jsonl)
    train_rows, valid_rows = split_simple(rows, test_size=0.12, seed=seed)

    Xtr, Ytr = make_regression_dataset(train_rows, y_keys)
    Xva, Yva = make_regression_dataset(valid_rows, y_keys)
    train_ds = Dataset.from_dict({"text": Xtr, "targets": Ytr})
    valid_ds = Dataset.from_dict({"text": Xva, "targets": Yva})
    ds = DatasetDict({"train": train_ds, "validation": valid_ds})

    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(batch):
        enc = tok(batch["text"], truncation=True, padding="max_length", max_length=max_len)
        enc["labels"] = batch["targets"]  # we will convert to tensor later
        return enc

    enc = ds.map(_pp, batched=True, remove_columns=["text","targets"])
    # figure token_type_ids availability
    cols = ["input_ids","attention_mask","labels"]
    if "token_type_ids" in enc["train"].features:
        cols.append("token_type_ids")
    enc.set_format(type="torch", columns=cols)

    model = BertRegressor(model_name, out_dim=len(y_keys))

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
        seed=seed,
    )

    def compute_metrics_reg(p):
        preds = p.predictions
        y = p.label_ids
        mse = mean_squared_error(y, preds)
        return {"mse": mse, "rmse": mse**0.5}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc["train"],
        eval_dataset=enc["validation"],
        tokenizer=tok,
        compute_metrics=compute_metrics_reg,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=20)]
    )

    trainer.train()
    eval_res = trainer.evaluate()

    final_dir = os.path.join(out_dir, "final_model")
    os.makedirs(final_dir, exist_ok=True)
    trainer.save_model(final_dir)
    tok.save_pretrained(final_dir)

    # Simpan info dimensi target
    with open(os.path.join(out_dir, "target_map.json"), "w", encoding="utf-8") as f:
        json.dump({"y_keys": y_keys}, f, ensure_ascii=False, indent=2)

    return final_dir, eval_res, y_keys


# -------------------------------------------------
# (3) LATIH SEMUA HEAD
# -------------------------------------------------

paths = {
    "intent":                  (f"{DATA_DIR}/train_intent.jsonl",                "multiclass", {"label_key":"label"}),
    "scope":                   (f"{DATA_DIR}/train_scope.jsonl",                 "multiclass", {"label_key":"label"}),
    "timecombo":               (f"{DATA_DIR}/train_timecombo.jsonl",             "multiclass", {"label_key":"label"}),
    "whatif_flag":             (f"{DATA_DIR}/train_whatif_flag.jsonl",           "multiclass", {"label_key":"label"}),

    # threshold classification heads
    "threshold_metric":        (f"{DATA_DIR}/train_threshold_metric.jsonl",      "multiclass", {"label_key":"label"}),
    "threshold_operator":      (f"{DATA_DIR}/train_threshold_operator.jsonl",    "multiclass", {"label_key":"label"}),

    # regression heads
    "horizon":                 (f"{DATA_DIR}/train_horizon.jsonl",               "regress",   {"y_keys":["y_start","y_end"]}),
    "threshold_value":         (f"{DATA_DIR}/train_threshold_value.jsonl",       "regress",   {"y_keys":["y_val"]}),
}

models = {}

for head_name, (train_file, mode, extra) in paths.items():
    print(f"\n--- Training head: {head_name} ---")
    out_dir = os.path.join(OUT_ROOT, head_name)
    os.makedirs(out_dir, exist_ok=True)

    if mode == "multiclass":
        mdir, evres, i2l = train_multiclass_head(
            BASE_MODEL, train_file, out_dir,
            label_key=extra["label_key"],
            epochs=EPOCHS, lr=LR, batch=BATCH, max_len=MAX_LEN, seed=SEED
        )
        models[head_name] = {"dir": mdir, "eval": evres, "i2l": i2l}

    elif mode == "regress":
        mdir, evres, ykeys = train_regression_head(
            BASE_MODEL, train_file, out_dir,
            y_keys=extra["y_keys"],
            epochs=EPOCHS, lr=LR, batch=BATCH, max_len=MAX_LEN, seed=SEED
        )
        models[head_name] = {"dir": mdir, "eval": evres, "y_keys": ykeys}

print("\nTraining selesai semua head.")
print("Output root:", OUT_ROOT)


# -------------------------------------------------
# (4) RUNNER GABUNGAN BUAT INFERENCE
# -------------------------------------------------

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import json

class ForecastMultiRunner:
    def __init__(self, root_dir):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # load multiclass heads
        self.intent_tok, self.intent_m, self.intent_i2l = self._load_mc(os.path.join(root_dir,"intent","final_model"))
        self.scope_tok, self.scope_m, self.scope_i2l = self._load_mc(os.path.join(root_dir,"scope","final_model"))
        self.timec_tok, self.timec_m, self.timec_i2l = self._load_mc(os.path.join(root_dir,"timecombo","final_model"))
        self.wif_tok, self.wif_m, self.wif_i2l       = self._load_mc(os.path.join(root_dir,"whatif_flag","final_model"))

        self.thmet_tok, self.thmet_m, self.thmet_i2l = self._load_mc(os.path.join(root_dir,"threshold_metric","final_model"))
        self.thop_tok,  self.thop_m,  self.thop_i2l  = self._load_mc(os.path.join(root_dir,"threshold_operator","final_model"))

        # load regression heads
        self.hor_tok, self.hor_m, self.hor_ykeys     = self._load_reg(os.path.join(root_dir,"horizon","final_model"))
        self.thval_tok, self.thval_m, self.thval_y   = self._load_reg(os.path.join(root_dir,"threshold_value","final_model"))

    def _load_mc(self, model_dir):
        cfg = AutoConfig.from_pretrained(model_dir)
        tok = AutoTokenizer.from_pretrained(model_dir)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).eval().to(self.device)
        i2l = cfg.id2label
        return tok, mdl, i2l

    def _load_reg(self, model_dir):
        # load tokenizer
        tok = AutoTokenizer.from_pretrained(model_dir)
        # load custom BertRegressor
        # we re-init same arch used in training:
        reg = BertRegressor(BASE_MODEL, out_dim=len(json.load(open(os.path.join(os.path.dirname(model_dir),"target_map.json")))["y_keys"]))
        # load state_dict
        state_dict = torch.load(os.path.join(model_dir,"pytorch_model.bin"), map_location="cpu")
        reg.load_state_dict(state_dict)
        reg.eval().to(self.device)
        # load y_keys
        with open(os.path.join(os.path.dirname(model_dir),"target_map.json"), "r", encoding="utf-8") as f:
            meta = json.load(f)
        return tok, reg, meta["y_keys"]

    @staticmethod
    def _decode(i2l, idx: int):
        # handle dict of {0:'labelA',1:'labelB',...}
        if isinstance(i2l, dict):
            return i2l.get(idx) or i2l.get(str(idx)) or f"label_{idx}"
        try:
            return i2l[idx]
        except:
            return f"label_{idx}"

    @torch.no_grad()
    def _run_mc(self, tok, mdl, i2l, text, max_len=128):
        enc = tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        logits = mdl(**enc).logits.squeeze(0)
        probs = torch.softmax(logits, dim=0)
        idx = int(torch.argmax(probs).item())
        lab = self._decode(i2l, idx)
        conf = float(probs[idx].item())
        return lab, conf

    @torch.no_grad()
    def _run_reg(self, tok, mdl, ykeys, text, max_len=128):
        enc = tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        out = mdl(**enc)
        preds = out["logits"].squeeze(0).cpu().numpy().tolist()
        return {ykeys[i]: float(preds[i]) for i in range(len(ykeys))}

    def predict(self, text: str, max_len=128):
        # 1. intent
        intent_lab, intent_conf = self._run_mc(self.intent_tok, self.intent_m, self.intent_i2l, text, max_len)
        # 2. scope
        scope_lab, scope_conf   = self._run_mc(self.scope_tok, self.scope_m, self.scope_i2l, text, max_len)
        # 3. time combo
        timec_lab, timec_conf   = self._run_mc(self.timec_tok, self.timec_m, self.timec_i2l, text, max_len)
        #    Pecah time combo biar downstream gampang
        time_granularity, anchor_time_type, time_complexity, horizon_type = timec_lab.split("|")

        # 4. horizon regression
        horizon_pred            = self._run_reg(self.hor_tok, self.hor_m, self.hor_ykeys, text, max_len)
        # 5. threshold stuff
        thr_metric_lab, thr_metric_conf = self._run_mc(self.thmet_tok, self.thmet_m, self.thmet_i2l, text, max_len)
        thr_oper_lab,  thr_oper_conf    = self._run_mc(self.thop_tok,  self.thop_m,  self.thop_i2l,  text, max_len)
        thr_value_pred           = self._run_reg(self.thval_tok, self.thval_m, self.thval_y, text, max_len)

        # 6. what-if flag
        whatif_lab, whatif_conf = self._run_mc(self.wif_tok, self.wif_m, self.wif_i2l, text, max_len)
        is_whatif = (whatif_lab == "IS_WHAT_IF")

        # Route hints downstream → siapa yang ngerjain setelah ini?
        route = []
        # scope route
        if scope_lab == "ALL":
            route.append("aggregate_all_machines")
        else:
            route.append("machine_entity_extractor")

        # intent route
        if intent_lab == "FORECAST_VALUE":
            route.append("forecast_numeric_pipeline")
        elif intent_lab == "FORECAST_TREND":
            route.append("forecast_trend_pipeline")
        elif intent_lab == "THRESHOLD_TIME":
            route.append("threshold_time_pipeline")
        elif intent_lab == "WHAT_IF_FORECAST":
            route.append("counterfactual_simulation_pipeline")

        # time handling route
        route.append(f"time_parse_{time_granularity.lower()}_{time_complexity.lower()}")

        if is_whatif:
            route.append("inject_counterfactual_state")

        return {
            "intent": {
                "label": intent_lab,
                "confidence": round(intent_conf,4)
            },
            "scope": {
                "label": scope_lab,
                "confidence": round(scope_conf,4)
            },
            "time": {
                "time_granularity": time_granularity,
                "anchor_time_type": anchor_time_type,
                "time_complexity": time_complexity,
                "horizon_type": horizon_type,
                "confidence": round(timec_conf,4)
            },
            "horizon_minutes": horizon_pred,  # {"y_start": ~3.0, "y_end": ~48.0} or {"y_val": ...}
            "threshold": {
                "metric": {"label": thr_metric_lab, "confidence": round(thr_metric_conf,4)},
                "operator": {"label": thr_oper_lab, "confidence": round(thr_oper_conf,4)},
                "value_predicted": thr_value_pred  # {"y_val": ~80.0}
            },
            "what_if": {
                "flag": whatif_lab,
                "is_counterfactual": is_whatif,
                "confidence": round(whatif_conf,4)
            },
            "route_hints": route
        }

# ============ (5) DEMO INFERENCE ============
runner = ForecastMultiRunner(OUT_ROOT)

examples = [
    "nilai mesin XP888A 2 jam lagi berapa?",
    "coba prediksi semua mesin apakah akan ada anomali 2 jam kedepan?",
    "kapan XP888A bakal turun di bawah 70 derajat? kalau iya jam berapa pertama kalinya?",
    "andaikan data XP888A kemarin jam 10:00 gue ubah jadi 95, efeknya apa 2 jam lagi?",
    "ramalan performa semua mesin untuk 30 menit ke depan, interval 3 menit"
]

print("\n=== DEMO PREDIKSI ===")
for q in examples:
    print("\nQ:", q)
    print(runner.predict(q))



--- Training head: intent ---


Map:   0%|          | 0/5280 [00:00<?, ? examples/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_695/2471480463.py:125: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.000400,0.000252,1.000000,1.000000
2,0.000200,0.000112,1.000000,1.000000
3,0.000100,0.000068,1.000000,1.000000
4,0.000100,0.000047,1.000000,1.000000
5,0.000000,0.000035,1.000000,1.000000
6,0.000000,0.000027,1.000000,1.000000
7,0.000000,0.000022,1.000000,1.000000
8,0.000000,0.000018,1.000000,1.000000
9,0.000000,0.000015,1.000000,1.000000
10,0.000000,0.000013,1.000000,1.000000


...


--- Training head: whatif_flag ---


Map:   0%|          | 0/5280 [00:00<?, ? examples/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_695/2471480463.py:125: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.000100,0.000036,1.000000,1.000000
2,0.000000,0.000016,1.000000,1.000000
3,0.000000,0.000010,1.000000,1.000000
4,0.000000,0.000007,1.000000,1.000000
5,0.000000,0.000005,1.000000,1.000000
6,0.000000,0.000004,1.000000,1.000000
7,0.000000,0.000003,1.000000,1.000000
8,0.000000,0.000003,1.000000,1.000000
9,0.000000,0.000002,1.000000,1.000000
10,0.000000,0.000002,1.000000,1.000000


In [31]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import json

import os, json, random, numpy as np, warnings
from pathlib import Path
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, mean_squared_error
from sklearn.model_selection import StratifiedShuffleSplit

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
DATA_DIR   = "/workspace/next"   # ganti sesuai folder dataset hasil generator tadi
OUT_ROOT   = "/workspace/runs_forecast_heads_v1"
BASE_MODEL = "indobenchmark/indobert-base-p1" 
from torch import nn

class BertRegressor(nn.Module):
    """
    Simple regression head on top of base transformer.
    We'll reuse pretrained model's encoder via AutoModelForSequenceClassification's config trick:
    Actually easier: load AutoModelForSequenceClassification then replace classifier.
    BUT we'll just build from AutoModelForSequenceClassification and set num_labels = out_dim,
    and use MSE loss manually in Trainer via custom compute_loss.
    """
    def __init__(self, base_name, out_dim=1):
        super().__init__()
        from transformers import AutoModel
        self.backbone = AutoModel.from_pretrained(base_name)
        hidden = self.backbone.config.hidden_size
        self.reg_head = nn.Linear(hidden, out_dim)
        self.config = self.backbone.config
        self.out_dim = out_dim

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # use CLS token
        cls = out.last_hidden_state[:,0,:]
        preds = self.reg_head(cls)
        loss = None
        if labels is not None:
            loss_fn = nn.MSELoss()
            loss = loss_fn(preds, labels.float())
        return {"loss": loss, "logits": preds}

class ForecastMultiRunner:
    def __init__(self, root_dir):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # load multiclass heads
        self.intent_tok, self.intent_m, self.intent_i2l = self._load_mc(os.path.join(root_dir,"intent","final_model"))
        self.scope_tok, self.scope_m, self.scope_i2l = self._load_mc(os.path.join(root_dir,"scope","final_model"))
        self.timec_tok, self.timec_m, self.timec_i2l = self._load_mc(os.path.join(root_dir,"timecombo","final_model"))
        self.wif_tok, self.wif_m, self.wif_i2l       = self._load_mc(os.path.join(root_dir,"whatif_flag","final_model"))

        self.thmet_tok, self.thmet_m, self.thmet_i2l = self._load_mc(os.path.join(root_dir,"threshold_metric","final_model"))
        self.thop_tok,  self.thop_m,  self.thop_i2l  = self._load_mc(os.path.join(root_dir,"threshold_operator","final_model"))

        # load regression heads
        self.hor_tok, self.hor_m, self.hor_ykeys     = self._load_reg(os.path.join(root_dir,"horizon","final_model"))
        self.thval_tok, self.thval_m, self.thval_y   = self._load_reg(os.path.join(root_dir,"threshold_value","final_model"))

    def _load_mc(self, model_dir):
        cfg = AutoConfig.from_pretrained(model_dir)
        tok = AutoTokenizer.from_pretrained(model_dir)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).eval().to(self.device)
        i2l = cfg.id2label
        return tok, mdl, i2l

    # def _load_reg(self, model_dir):
    #     # load tokenizer
    #     tok = AutoTokenizer.from_pretrained(model_dir)
    #     # load custom BertRegressor
    #     # we re-init same arch used in training:
    #     reg = BertRegressor(BASE_MODEL, out_dim=len(json.load(open(os.path.join(os.path.dirname(model_dir),"target_map.json")))["y_keys"]))
    #     # load state_dict
    #     state_dict = torch.load(os.path.join(model_dir,"pytorch_model.bin"), map_location="cpu")
    #     reg.load_state_dict(state_dict)
    #     reg.eval().to(self.device)
    #     # load y_keys
    #     with open(os.path.join(os.path.dirname(model_dir),"target_map.json"), "r", encoding="utf-8") as f:
    #         meta = json.load(f)
    #     return tok, reg, meta["y_keys"]

    def _load_reg(self, model_dir):
    # 1. load tokenizer
        tok = AutoTokenizer.from_pretrained(model_dir)
    
        # 2. cari target y_keys
        with open(os.path.join(os.path.dirname(model_dir), "target_map.json")) as f:
            target_info = json.load(f)
        y_keys = target_info["y_keys"]
    
        # 3. bikin model skeleton dengan output dim yang sesuai
        reg = BertRegressor(BASE_MODEL, out_dim=len(y_keys))
    
        # 4. load weight
        safetensor_path = os.path.join(model_dir, "model.safetensors")
    
        import safetensors.torch as st
        state_dict = st.load_file(safetensor_path, device="cpu")
    
        reg.load_state_dict(state_dict)
    
        # 5. eval mode + device
        reg.eval().to(self.device)
    
        return tok, reg, y_keys

    @staticmethod
    def _decode(i2l, idx: int):
        # handle dict of {0:'labelA',1:'labelB',...}
        if isinstance(i2l, dict):
            return i2l.get(idx) or i2l.get(str(idx)) or f"label_{idx}"
        try:
            return i2l[idx]
        except:
            return f"label_{idx}"

    @torch.no_grad()
    def _run_mc(self, tok, mdl, i2l, text, max_len=128):
        enc = tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        logits = mdl(**enc).logits.squeeze(0)
        probs = torch.softmax(logits, dim=0)
        idx = int(torch.argmax(probs).item())
        lab = self._decode(i2l, idx)
        conf = float(probs[idx].item())
        return lab, conf

    @torch.no_grad()
    def _run_reg(self, tok, mdl, ykeys, text, max_len=128):
        enc = tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        out = mdl(**enc)
        preds = out["logits"].squeeze(0).cpu().numpy().tolist()
        return {ykeys[i]: float(preds[i]) for i in range(len(ykeys))}

    def predict(self, text: str, max_len=128):
        # 1. intent
        intent_lab, intent_conf = self._run_mc(self.intent_tok, self.intent_m, self.intent_i2l, text, max_len)
        # 2. scope
        scope_lab, scope_conf   = self._run_mc(self.scope_tok, self.scope_m, self.scope_i2l, text, max_len)
        # 3. time combo
        timec_lab, timec_conf   = self._run_mc(self.timec_tok, self.timec_m, self.timec_i2l, text, max_len)
        #    Pecah time combo biar downstream gampang
        time_granularity, anchor_time_type, time_complexity, horizon_type = timec_lab.split("|")

        # 4. horizon regression
        horizon_pred            = self._run_reg(self.hor_tok, self.hor_m, self.hor_ykeys, text, max_len)
        # 5. threshold stuff
        thr_metric_lab, thr_metric_conf = self._run_mc(self.thmet_tok, self.thmet_m, self.thmet_i2l, text, max_len)
        thr_oper_lab,  thr_oper_conf    = self._run_mc(self.thop_tok,  self.thop_m,  self.thop_i2l,  text, max_len)
        thr_value_pred           = self._run_reg(self.thval_tok, self.thval_m, self.thval_y, text, max_len)

        # 6. what-if flag
        whatif_lab, whatif_conf = self._run_mc(self.wif_tok, self.wif_m, self.wif_i2l, text, max_len)
        is_whatif = (whatif_lab == "IS_WHAT_IF")

        # Route hints downstream → siapa yang ngerjain setelah ini?
        route = []
        # scope route
        if scope_lab == "ALL":
            route.append("aggregate_all_machines")
        else:
            route.append("machine_entity_extractor")

        # intent route
        if intent_lab == "FORECAST_VALUE":
            route.append("forecast_numeric_pipeline")
        elif intent_lab == "FORECAST_TREND":
            route.append("forecast_trend_pipeline")
        elif intent_lab == "THRESHOLD_TIME":
            route.append("threshold_time_pipeline")
        elif intent_lab == "WHAT_IF_FORECAST":
            route.append("counterfactual_simulation_pipeline")

        # time handling route
        route.append(f"time_parse_{time_granularity.lower()}_{time_complexity.lower()}")

        if is_whatif:
            route.append("inject_counterfactual_state")

        return {
            "intent": {
                "label": intent_lab,
                "confidence": round(intent_conf,4)
            },
            "scope": {
                "label": scope_lab,
                "confidence": round(scope_conf,4)
            },
            "time": {
                "time_granularity": time_granularity,
                "anchor_time_type": anchor_time_type,
                "time_complexity": time_complexity,
                "horizon_type": horizon_type,
                "confidence": round(timec_conf,4)
            },
            "horizon_minutes": horizon_pred,  # {"y_start": ~3.0, "y_end": ~48.0} or {"y_val": ...}
            "threshold": {
                "metric": {"label": thr_metric_lab, "confidence": round(thr_metric_conf,4)},
                "operator": {"label": thr_oper_lab, "confidence": round(thr_oper_conf,4)},
                "value_predicted": thr_value_pred  # {"y_val": ~80.0}
            },
            "what_if": {
                "flag": whatif_lab,
                "is_counterfactual": is_whatif,
                "confidence": round(whatif_conf,4)
            },
            "route_hints": route
        }

import re
import math

KNOWN_MACHINES = ["XP888A","PQ667","ZT901B","LM442","KJ3-A","ALPHA01","LINE-7"]

def _extract_machines(user_text, scope_label):
    """
    Ambil nama mesin dari teks user.
    - kalau scope ALL -> ["ALL"]
    - kalau ada beberapa mesin disebut -> return list unik
    - kalau tidak ada match tapi scope SINGLE/MULTI -> ["UNKNOWN"]
    """
    if scope_label == "ALL":
        return ["ALL"]

    found = []
    for m in KNOWN_MACHINES:
        # case-insensitive contain check, also handle things like "XP888A," or "XP888A?" dsb
        pattern = r"\b" + re.escape(m) + r"\b"
        if re.search(pattern, user_text, flags=re.IGNORECASE):
            found.append(m)
    found = list(dict.fromkeys(found))  # dedup, preserve order

    if not found:
        return ["UNKNOWN"]
    return found

def _snap_single_point(start_pred, end_pred):
    """
    Model kadang kasih 92 vs 101 menit padahal SINGLE_POINT.
    Ambil rata-rata dan bulatkan.
    """
    avg_val = (float(start_pred) + float(end_pred)) / 2.0
    avg_val = round(avg_val)
    return avg_val, avg_val

def _parse_explicit_horizon(user_text, time_info):
    """
    Coba deteksi horizon dari teks user.
    Return (start_min, end_min, used_rule:bool) 
    """
    tcomp = time_info["time_complexity"]

    txt = user_text.lower()

    # pattern "<N> jam"
    m_jam = re.search(r"(\d+)\s*(jam|jm)\b", txt)
    # pattern "<N> menit" / mnt
    m_menit = re.search(r"(\d+)\s*(menit|mnt)\b", txt)

    # Helper to convert matched number
    def _to_int(m):
        try:
            return int(m.group(1))
        except:
            return None

    # CASE 1: menit-based horizon like "30 menit ke depan"
    if m_menit:
        n_min = _to_int(m_menit)
        if n_min is not None:
            # Check if wording is "ke depan", "sampai", "interval"
            # -> treat as RANGE 0..n_min
            if "ke depan" in txt or "kedepan" in txt or "sampai" in txt or "interval" in txt:
                start_min = 0
                end_min   = n_min
            else:
                # fallback generic:
                if tcomp == "SINGLE_POINT":
                    start_min = n_min
                    end_min   = n_min
                else:
                    start_min = 0
                    end_min   = n_min
            return start_min, end_min, True

    # CASE 2: jam-based horizon like "2 jam lagi" / "2 jam ke depan"
    if m_jam:
        n_hour = _to_int(m_jam)
        if n_hour is not None:
            total_min = n_hour * 60

            # kalau SINGLE_POINT → t+N jam
            if tcomp == "SINGLE_POINT":
                start_min = total_min
                end_min   = total_min
            else:
                # RANGE_REL → 0..N jam
                start_min = 0
                end_min   = total_min
            return start_min, end_min, True

    # CASE 3: "dalam <N> menit ke depan"
    m_dlm = re.search(r"dalam\s+(\d+)\s*(menit|mnt)", txt)
    if m_dlm:
        n_min = int(m_dlm.group(1))
        start_min = 0
        end_min   = n_min
        return start_min, end_min, True

    # CASE 4: phrases "X menit depan", "X menit lagi"
    m_simple_m = re.search(r"(\d+)\s*(menit|mnt)\s*(lagi|depan|kedepan)", txt)
    if m_simple_m:
        n_min = int(m_simple_m.group(1))
        if tcomp == "SINGLE_POINT":
            return n_min, n_min, True
        else:
            return 0, n_min, True

    # CASE 5: phrases "X jam depan", "X jam kedepan"
    m_simple_h = re.search(r"(\d+)\s*(jam|jm)\s*(lagi|depan|kedepan)", txt)
    if m_simple_h:
        n_h = int(m_simple_h.group(1))
        tot = n_h * 60
        if tcomp == "SINGLE_POINT":
            return tot, tot, True
        else:
            return 0, tot, True

    # kalau tidak ada match → return None
    return None, None, False
def extract_all_requested_minutes(user_text: str):
    """
    Cari semua 'N menit', 'N jam', 'N mnt', dst.
    Return list menit (int).
    """
    # contoh pola: "48 menit", "24 menit", "12 menit", "2 jam", "1 jam", "30 mnt"
    pattern = r'(\d+)\s*(menit|mnt|minute|min|jam|hour|h)'
    hits = re.findall(pattern, user_text.lower())

    mins = []
    for num_str, unit in hits:
        val = int(num_str)
        if "jam" in unit or unit in ["hour","h"]:
            mins.append(val * 60)
        else:
            mins.append(val)
    # unique + sort
    mins = sorted(list({m for m in mins}))
    return mins
    
def interpret_query(user_text: str, runner_output: dict):
    """
    user_text      : string original query user
    runner_output  : dict hasil ForecastMultiRunner.predict()
    return         : dict final NormalizedSpec
    """

    # 1. ambil komponen dari runner_output
    intent_lab   = runner_output["intent"]["label"]
    scope_lab    = runner_output["scope"]["label"]
    time_info    = runner_output["time"]  # dict
    horizon_pred = runner_output["horizon_minutes"]  # {'y_start':..., 'y_end':...}
    whatif_info  = runner_output["what_if"]
    thr_info     = runner_output["threshold"]

    # 2. ekstrak mesin
    machines = _extract_machines(user_text, scope_lab)

    # 3. normalize horizon
    #    3a. coba parse langsung dari teks
    explicit_start, explicit_end, ok_rule = _parse_explicit_horizon(user_text, time_info)

    if ok_rule:
        final_start = explicit_start
        final_end   = explicit_end
        horizon_source = "text_rule"
    else:
        # fallback: pakai output model
        y_s = float(horizon_pred.get("y_start", -1))
        y_e = float(horizon_pred.get("y_end", -1))

        # rapihin SINGLE_POINT
        if time_info["time_complexity"] == "SINGLE_POINT":
            y_s, y_e = _snap_single_point(y_s, y_e)

        final_start = max(0, round(y_s)) if y_s >= 0 else -1
        final_end   = max(final_start, round(y_e)) if y_e >= 0 else -1
        horizon_source = "model_regressor"

    # 4. threshold object
    threshold_spec = None
    if intent_lab == "THRESHOLD_TIME":
        met = thr_info["metric"]["label"]
        op  = thr_info["operator"]["label"]
        val_pred = thr_info["value_predicted"].get("y_val", -1)

        # map operator words → symbol
        if op == "RISE_ABOVE":
            op_symbol = ">"
        elif op == "DROP_BELOW":
            op_symbol = "<"
        else:
            op_symbol = None

        # threshold_spec hanya valid kalau metric!=NONE dan op_symbol!=None
        if met != "NONE" and op_symbol is not None and val_pred >= 0:
            threshold_spec = {
                "metric": met,
                "operator": op_symbol,
                "value": round(float(val_pred), 2)
            }

    # 5. counterfactual flag
    counterfactual_mode = whatif_info["is_counterfactual"]

    # 6. action type (routing high level)
    if intent_lab == "FORECAST_VALUE":
        action_type = "forecast_numeric"
    elif intent_lab == "FORECAST_TREND":
        action_type = "forecast_trend"
    elif intent_lab == "THRESHOLD_TIME":
        action_type = "threshold_eta"
    elif intent_lab == "WHAT_IF_FORECAST":
        action_type = "counterfactual_forecast"
    else:
        action_type = "unknown"

    # 7. build final normalized spec
    spec = {
        "action_type": action_type,
        "intent": intent_lab,
        "scope": scope_lab,
        "machines": machines,  # list
        "time": {
            "granularity": time_info["time_granularity"],
            "anchor_time_type": time_info["anchor_time_type"],
            "time_complexity": time_info["time_complexity"],
            "horizon_type": time_info["horizon_type"],
            "window_minutes": {
                "start_min": final_start,
                "end_min": final_end,
                "source": horizon_source,
            }
        },
        "threshold": threshold_spec,
        "counterfactual": {
            "enabled": counterfactual_mode,
            # kamu bisa isi extra extraction di sini nanti (nilai override dsb)
            "raw_query": user_text if counterfactual_mode else None
        },
        "route_hints": runner_output["route_hints"]
    }
    all_mins = extract_all_requested_minutes(user_text)
    if len(all_mins) > 1:
        # simpan list multi target
        spec['time']['multi_targets_minutes'] = all_mins
    else:
        spec['time']['multi_targets_minutes'] = None

    return spec


In [17]:
runner = ForecastMultiRunner(OUT_ROOT)

examples = [
    "nilai mesin XP888A 2 jam lagi berapa?",
    "coba prediksi semua mesin apakah akan ada anomali 2 jam kedepan?",
    "kapan XP888A bakal turun di bawah 70 derajat? kalau iya jam berapa pertama kalinya?",
    "andaikan data XP888A kemarin jam 10:00 gue ubah jadi 95, efeknya apa 2 jam lagi?",
    "ramalan performa semua mesin untuk 30 menit ke depan, interval 3 menit"
]

print("\n=== DEMO PREDIKSI ===")
for q in examples:
    print("\nQ:", q)
    runner_out=runner.predict(q)
    spec = interpret_query(q, runner_out)
    print(runner_out)
    print()
    print(spec)
    print()


=== DEMO PREDIKSI ===

Q: nilai mesin XP888A 2 jam lagi berapa?
{'intent': {'label': 'FORECAST_VALUE', 'confidence': 0.9998}, 'scope': {'label': 'SINGLE', 'confidence': 0.9999}, 'time': {'time_granularity': 'HOUR', 'anchor_time_type': 'NOW_START', 'time_complexity': 'SINGLE_POINT', 'horizon_type': 'FIXED_STEPS', 'confidence': 0.9986}, 'horizon_minutes': {'y_start': 92.41140747070312, 'y_end': 101.1116714477539}, 'threshold': {'metric': {'label': 'NONE', 'confidence': 0.9998}, 'operator': {'label': 'NONE', 'confidence': 0.9999}, 'value_predicted': {'y_val': -1.1409823894500732}}, 'what_if': {'flag': 'NORMAL', 'is_counterfactual': False, 'confidence': 1.0}, 'route_hints': ['machine_entity_extractor', 'forecast_numeric_pipeline', 'time_parse_hour_single_point']}

{'action_type': 'forecast_numeric', 'intent': 'FORECAST_VALUE', 'scope': 'SINGLE', 'machines': ['XP888A'], 'time': {'granularity': 'HOUR', 'anchor_time_type': 'NOW_START', 'time_complexity': 'SINGLE_POINT', 'horizon_type': 'FIXE

In [18]:
import numpy as np

HORIZON_MINUTES = {
    "y_step_3m": 3,
    "y_step_6m": 6,
    "y_step_12m": 12,
    "y_step_24m": 24,
    "y_step_48m": 48,
}

# kebalikan biar gampang panggil model dari menit:
HORIZON_BY_MIN = {v:k for k,v in HORIZON_MINUTES.items()}


def shift_lags(state_lags, new_value):
    """
    Geser lag window setelah kita 'bergerak' ke masa depan.
    Asumsi lag_1 = nilai paling baru, lag_2 = sebelumnya, dst.
    """
    new_state = state_lags.copy()
    # contoh kita cuma punya 3 lag
    new_state["y_lag_3"] = state_lags["y_lag_2"]
    new_state["y_lag_2"] = state_lags["y_lag_1"]
    new_state["y_lag_1"] = float(new_value)
    return new_state


def onestep_predict_minutes(forecaster, state_lags, horizon_min):
    """
    Pakai model horizon tertentu (misal 12 menit) untuk prediksi value masa depan relatif dari state_lags sekarang.
    Return pred_value (float).
    """
    # model key
    hk = HORIZON_BY_MIN[horizon_min]
    pred_val = forecaster.predict_single_horizon(hk, state_lags)
    return pred_val


def choose_hop_sequence(target_minutes):
    """
    Ini adalah tempat kita encode strategi 'pakai 12 step lalu 48 step lalu ...'
    Kamu bisa tweak sesuai kebijakan produksi.
    Beberapa contoh rule (heuristic):
    - <=6  -> [3,3]  (atau [6] kalo mau langsung)
    - <=12 -> [12]
    - <=24 -> [12,12] atau [24]
    - <=48 -> [24,24] atau [48]
    - <=60 -> [12,48]
    - >60  -> [12,48,48,12]   # contoh pattern kamu untuk ~120 menit
    Catatan: hasil total hop kita tidak harus persis target, tapi mendekati.
    Nanti kita hitung menit kumulatif biar keliatan sampai berapa menit sebenernya.
    """
    t = target_minutes
    if t <= 6:
        return [3,3] if t > 3 else [3]
    if t <= 12:
        return [12]
    if t <= 24:
        return [12,12] if t < 24 else [24]
    if t <= 48:
        return [24] if t <= 30 else [24,24] if t > 30 else [24]
    if t <= 60:
        return [12,48]
    # t > 60
    # pattern kamu buat 2 jam (120) adalah 12,48,48,12
    # kalau t super besar, kita bisa ulangin pola ini
    # simple approach:
    hops = []
    remaining = t
    while remaining > 60:
        hops.extend([12,48])  # dua langkah, total 60
        remaining -= 60
    # sisa 'remaining' sekarang <=60
    if remaining > 48:
        hops.append(48); remaining -= 48
    elif remaining > 24:
        hops.append(24); remaining -= 24
    elif remaining > 12:
        hops.append(12); remaining -= 12
    # kalau masih ada sisa kecil (<12), tambahin 3/6/12 kecil
    if remaining > 6:
        hops.append(6); remaining -= 6
    if remaining > 0:
        hops.append(3)
    return hops


def rollout_autoregressive(forecaster, init_state_lags, target_minutes):
    """
    Jalankan hop sequence sampai mendekati target_minutes.
    - forecaster: HorizonForecasterClassic
    - init_state_lags: dict {"y_lag_1":..., "y_lag_2":..., "y_lag_3":...}
    - target_minutes: misal 120
    Return:
      {
        "trajectory": [
            {"t_accum": 12, "pred": v12},
            {"t_accum": 60, "pred": v60},
            {"t_accum": 108, "pred": v108},
            {"t_accum": 120, "pred": v120}
        ],
        "final_pred": v_last,
        "t_final": t_last
      }
    """

    hops = choose_hop_sequence(target_minutes)
    traj = []
    state = init_state_lags.copy()
    t_accum = 0

    for hop_min in hops:
        pred_val = onestep_predict_minutes(forecaster, state, hop_min)
        t_accum += hop_min
        traj.append({"t_accum": t_accum, "pred": pred_val, "hop": hop_min})
        # update lags untuk langkah berikutnya
        state = shift_lags(state, pred_val)

    # final value = pred setelah hop terakhir
    final_pred = traj[-1]["pred"]
    t_final    = traj[-1]["t_accum"]

    return {
        "trajectory": traj,
        "final_pred": final_pred,
        "t_final": t_final
    }


In [19]:
def execute_spec(spec, forecaster, context_dict):
    action = spec["action_type"]
    tinfo  = spec["time"]
    wc     = tinfo["window_minutes"]
    start_m = wc["start_min"]
    end_m   = wc["end_min"]

    # FORECAST_NUMERIC & COUNTERFACTUAL_FORECAST → numeric future
    if action in ["forecast_numeric", "counterfactual_forecast", "forecast_trend"]:
        tc = tinfo["time_complexity"]

        if tc == "SINGLE_POINT":
            # gunakan autoregressive rollout sampai end_m menit
            rollout = rollout_autoregressive(forecaster, context_dict, end_m)
            return {
                "type": action,
                "machines": spec["machines"],
                "mode": "autoregressive_chain",
                "target_request_min": end_m,
                "result_single": {
                    "final_pred": rollout["final_pred"],
                    "t_final": rollout["t_final"],
                    "trajectory": rollout["trajectory"]
                },
                "counterfactual": spec["counterfactual"] if action=="counterfactual_forecast" else None
            }

        else:
            # RANGE_REL / RANGE_FUTURE
            # untuk range kita bisa:
            # - generate grid titik waktu
            # - untuk setiap titik waktu jalankan rollout_autoregressive sendiri sampai titik itu
            # Ini mahal tapi jelas. (bisa di-cache)
            targets = build_range_targets(start_m, end_m)
            timeline = []
            for tmin in targets:
                rollout = rollout_autoregressive(forecaster, context_dict, tmin)
                timeline.append({
                    "minute_ahead": tmin,
                    "pred_value": rollout["final_pred"],
                    "t_simulated": rollout["t_final"],
                    "trajectory": rollout["trajectory"]
                })

            return {
                "type": action,
                "machines": spec["machines"],
                "mode": "autoregressive_chain",
                "window": [start_m, end_m],
                "timeline": timeline,
                "counterfactual": spec["counterfactual"] if action=="counterfactual_forecast" else None
            }

    # THRESHOLD_TIME tetap sama seperti sebelumnya
    if action == "threshold_eta":
        return {
            "type": "threshold_eta",
            "machines": spec["machines"],
            "threshold": spec["threshold"],
            "eta_minutes_range": [wc["start_min"], wc["end_min"]],
        }

    return {
        "type": "unsupported",
        "raw_spec": spec
    }


In [40]:
############################################################
# 0. IMPORT & PREP
############################################################

import re, json, numpy as np
from pathlib import Path

# === asumsi ini sudah ada dari step sebelumnya ===
# from your_nlu_module import ForecastMultiRunner, interpret_query
# from your_training_module import HorizonForecasterClassic
# plus torch / joblib dll sudah diimport

############################################################
# 1. HORIZON FORECASTER (classical Linear+RF residual)
############################################################

import joblib

HORIZON_MINUTES = {
    "y_step_3m": 3,
    "y_step_6m": 6,
    "y_step_12m": 12,
    "y_step_24m": 24,
    "y_step_48m": 48,
}
HORIZON_BY_MIN = {v:k for k,v in HORIZON_MINUTES.items()}

class HorizonForecasterClassic:
    """
    Loader + predictor untuk model classical combo (Linear + RF residual)
    yang kamu simpan per horizon di outputs_<timestamp>/<horizon>/
    """
    def __init__(self, run_root):
        self.run_root = Path(run_root)
        self.cache = {}  # { "y_step_12m": {lin, rf, use_lags} }

    def _load_horizon(self, horizon_key):
        if horizon_key in self.cache:
            return self.cache[horizon_key]

        hz_dir = self.run_root / horizon_key
        lin = joblib.load(hz_dir / f"{horizon_key}_linear.joblib")
        rf  = joblib.load(hz_dir / f"{horizon_key}_rf_residual.joblib")

        cfg = json.loads((hz_dir / f"{horizon_key}_config.json").read_text())
        use_lags = cfg["use_lags"]  # contoh: ["y_lag_1","y_lag_2","y_lag_3"]

        self.cache[horizon_key] = {
            "lin": lin,
            "rf": rf,
            "use_lags": use_lags,
        }
        return self.cache[horizon_key]

    def predict_single_horizon(self, horizon_key, state_lags):
        """
        horizon_key: 'y_step_12m' dll
        state_lags: dict fitur realtime saat ini:
            {
                "y_lag_1": <float latest value>,
                "y_lag_2": <float prev>,
                "y_lag_3": <float prevprev>,
                ...
            }
        return float pred_combo
        """
        bundle = self._load_horizon(horizon_key)
        use_lags = bundle["use_lags"]

        X_now = np.array([[ state_lags[feat] for feat in use_lags ]], dtype=float)

        y_lin = bundle["lin"].predict(X_now)[0]
        y_res = bundle["rf"].predict(X_now)[0]
        y_combo = float(y_lin + y_res)
        return y_combo


############################################################
# 2. AUTOREGRESSIVE ROLLOUT ENGINE
############################################################

def shift_lags(state_lags, new_value):
    """
    Geser lag window setelah kita melangkah ke masa depan.
    Asumsi:
    - y_lag_1 = nilai terkini
    - y_lag_2 = nilai sebelumnya
    - y_lag_3 = nilai sebelumnya lagi
    Kita inject prediksi baru sebagai y_lag_1 berikutnya.
    """
    new_state = state_lags.copy()
    # geser ke belakang
    if "y_lag_3" in new_state:
        new_state["y_lag_3"] = state_lags["y_lag_2"]
    if "y_lag_2" in new_state:
        new_state["y_lag_2"] = state_lags["y_lag_1"]
    # y_lag_1 update dengan prediksi baru
    new_state["y_lag_1"] = float(new_value)
    return new_state

def onestep_predict_minutes(forecaster, state_lags, horizon_min):
    """
    Pakai model horizon_min (3,6,12,24,48) untuk prediksi nilai future.
    """
    hk = HORIZON_BY_MIN[horizon_min]
    return forecaster.predict_single_horizon(hk, state_lags)

 
def choose_hop_sequence_cover(target_minutes, already_covered):
    """
    Balikin list hops baru untuk maju dari already_covered
    sampai >= target_minutes.
    Tidak boleh berhenti di bawah target.
    """
    hops = []
    t = already_covered

    while t < target_minutes:
        remaining = target_minutes - t

        # mode adaptif
        if remaining <= 12:
            candidates = [3,6,12,24,48]
        else:
            candidates = [48,24,12,6,3]

        chosen = None
        # pilih kandidat terkecil yang masih >= remaining
        for c in candidates:
            if c >= remaining:
                chosen = c
                break

        if chosen is None:
            # tidak ada single hop yang >= remaining,
            # ambil kandidat terbesar biar cepet maju
            chosen = candidates[-1] if remaining <= 12 else candidates[0]

        hops.append(chosen)
        t += chosen

    return hops


def choose_hop_sequence(target_minutes):
    """
    Heuristik sequence hop untuk autoregressive chain.
    Bisa (dan harus) kamu tweak sesuai domain.
    - <=6  -> [3,3] (kasih dua hop 3 menit)
    - <=12 -> [12]
    - <=24 -> [12,12] kalau <24, atau [24] kalau pas 24
    - <=48 -> [24,24] kalau >24, else [24]
    - <=60 -> [12,48]
    - >60  -> pecah jadi blok 60 menit pakai [12,48] berulang,
              lalu sisa terakhir dihajar pakai potongan yang mendekati.
    Contoh 120 menit → [12,48,48,12] ~ 120m total.
    """
    t = target_minutes
    if t <= 6:
        return [3,3] if t > 3 else [3]
    if t <= 12:
        return [12]
    if t <= 24:
        if t < 24:
            return [12,12]
        else:
            return [24]
    if t <= 48:
        if t <= 30:
            return [24]
        else:
            return [48]
    if t <= 60:
        return [12,48]

    # t > 60 → kita terus kurangi 60 pakai blok [12,48]
    hops = []
    remaining = t
    while remaining > 60:
        hops.extend([12,48])  # ini total 60 per loop
        remaining -= 60
    # handle sisa <=60
    if remaining > 48:
        hops.append(48); remaining -= 48
    elif remaining > 24:
        hops.append(24); remaining -= 24
    elif remaining > 12:
        hops.append(12); remaining -= 12
    # kalau masih ada sisa kecil, tambahin 6/3
    if remaining > 6:
        hops.append(6); remaining -= 6
    if remaining > 0:
        hops.append(3)
    return hops
def predict_hop_minutes(forecaster, state_lags, hop_min):
    """
    1 hop = prediksi ke depan sejauh hop_min menit pakai horizon model terdekat.
    hop_min HARUS salah satu [3,6,12,24,48].
    """
    horizon_key = HORIZON_BY_MIN[hop_min]
    return forecaster.predict_single_horizon(horizon_key, state_lags)
    
def rollout_incremental_horizons(forecaster, init_state_lags, targets_minutes_sorted):
    """
    targets_minutes_sorted: ex [12,24,48] ATAU [3,24,50,120]
    Jalan maju stepwise, simpan snapshot untuk tiap target.
    """
    results = {}
    state = init_state_lags.copy()
    t_accum = 0
    traj_global = []

    for tgt in targets_minutes_sorted:
        extra_hops = choose_hop_sequence_cover(tgt, t_accum)

        for hop_min in extra_hops:
            y_pred = predict_hop_minutes(forecaster, state, hop_min)
            t_accum += hop_min
            traj_global.append({
                "t_accum": t_accum,
                "hop": hop_min,
                "pred": y_pred
            })
            state = shift_lags(state, y_pred)

        # simpan hasil setelah kita minimal sampai target ini
        results[tgt] = {
            "pred_value": traj_global[-1]["pred"],
            "t_effective": t_accum,
            "trajectory_so_far": list(traj_global),  # copy for debug/audit
        }

    return results

    
def rollout_autoregressive(forecaster, init_state_lags, target_minutes):
    """
    Simulasi maju step-by-step sesuai hop sequence.
    Setelah tiap hop:
      - prediksi nilai masa depan
      - update lag dengan nilai prediksi tsb
    Output:
    {
      "trajectory": [
         {"t_accum": 12, "pred": v12, "hop": 12},
         {"t_accum": 60, "pred": v60, "hop": 48},
         ...
      ],
      "final_pred": v_last,
      "t_final": t_last
    }
    """
    hops = choose_hop_sequence(target_minutes)
    traj = []
    state = init_state_lags.copy()
    t_accum = 0

    for hop_min in hops:
        val_pred = onestep_predict_minutes(forecaster, state, hop_min)
        t_accum += hop_min
        traj.append({
            "t_accum": t_accum,
            "hop": hop_min,
            "pred": val_pred
        })
        # update kondisi jadi prediksi terakhir
        state = shift_lags(state, val_pred)

    return {
        "trajectory": traj,
        "final_pred": traj[-1]["pred"],
        "t_final": traj[-1]["t_accum"]
    }

def build_range_targets(start_min, end_min):
    """
    Untuk kasus RANGE (misal 0..30 menit ke depan),
    kita mau beberapa titik penting.
    Strategi simple:
      - gunakan horizon default [3,6,12,24,48]
      - ambil yang jatuh di window
      - pastikan titik akhir end_min juga ada
    """
    base_pts = sorted(HORIZON_MINUTES.values())  # [3,6,12,24,48]
    pts = [m for m in base_pts if (m >= start_min and m <= end_min)]
    if end_min not in pts:
        pts.append(end_min)
    pts = sorted(list(set(pts)))
    return pts

def forecast_point_autoreg(spec, forecaster, context_state):
    """
    Untuk SINGLE_POINT (contoh: '2 jam lagi berapa?').
    """
    target_min = spec["time"]["window_minutes"]["end_min"]  # sudah disnap jadi 120 dll
    rollout = rollout_autoregressive(forecaster, context_state, target_min)
    return {
        "target_request_min": target_min,
        "final_pred": rollout["final_pred"],
        "t_final": rollout["t_final"],
        "trajectory": rollout["trajectory"]
    }

def forecast_range_autoreg(spec, forecaster, context_state):
    """
    Untuk RANGE_REL/RANGE_FUTURE (contoh: '30 menit ke depan, interval 3 menit').
    Kita generate beberapa titik dalam window, dan jalankan rollout_autoregressive terpisah untuk tiap titik.
    """
    w = spec["time"]["window_minutes"]
    start_m = w["start_min"]
    end_m   = w["end_min"]

    targets = build_range_targets(start_m, end_m)
    series = []
    for tmin in targets:
        r = rollout_autoregressive(forecaster, context_state, tmin)
        series.append({
            "minute_ahead": tmin,
            "pred_value": r["final_pred"],
            "t_final": r["t_final"],
            "trajectory": r["trajectory"]
        })
    return {
        "window": [start_m, end_m],
        "timeline": series
    }

############################################################
# 3. EXECUTOR UNTUK SPEC (gabung ke pipeline)
############################################################

def execute_spec(spec, forecaster, context_state):
    """
    spec         : hasil interpret_query
    forecaster   : HorizonForecasterClassic instance
    context_state: dict lag realtime untuk mesin tsb
                   ex {"y_lag_1":105.2,"y_lag_2":104.8,"y_lag_3":104.5}
    return       : dict hasil final inference siap dikonsumsi UI / downstream
    """

    action = spec["action_type"]
    tinfo  = spec["time"]
    tc     = tinfo["time_complexity"]

   
    time_info = spec["time"]
    multi_list = time_info.get("multi_targets_minutes")

    # 0. MULTI-HORIZON QUERY (misal "12, 24, 48 menit")
    if multi_list and len(multi_list) > 1:
        # pastikan ascending
        reqs = sorted(multi_list)

        mh_res = rollout_incremental_horizons(
            forecaster,
            context_state,
            reqs
        )

        return {
            "type": "forecast_multi_point",
            "machines": spec["machines"],
            "requested_minutes": reqs,
            "per_horizon": {
                str(t): {
                    "pred_value": mh_res[t]["pred_value"],
                    "t_effective": mh_res[t]["t_effective"]
                }
                for t in reqs
            }
        }
    # counterfactual / forecast_numeric / forecast_trend -> numeric forecast forward
    if action in ["forecast_numeric", "forecast_trend", "counterfactual_forecast"]:
        if tc == "SINGLE_POINT":
            out_single = forecast_point_autoreg(spec, forecaster, context_state)
            return {
                "type": action,
                "machines": spec["machines"],
                "mode": "autoregressive_chain",
                "result_single": out_single,
                "counterfactual": spec["counterfactual"] if action=="counterfactual_forecast" else None
            }
        else:
            out_range = forecast_range_autoreg(spec, forecaster, context_state)
            return {
                "type": action,
                "machines": spec["machines"],
                "mode": "autoregressive_chain",
                "result_range": out_range,
                "counterfactual": spec["counterfactual"] if action=="counterfactual_forecast" else None
            }

    # threshold question
    if action == "threshold_eta":
        wc = tinfo["window_minutes"]
        return {
            "type": "threshold_eta",
            "machines": spec["machines"],
            "threshold": spec["threshold"],
            "eta_minutes_range": [wc["start_min"], wc["end_min"]]
        }

    return {
        "type": "unsupported",
        "raw_spec": spec
    }



In [48]:
import pandas as pd
from datetime import datetime
import json

def load_raw_timeseries(csv_path="/workspace/df_raw2.csv"):
    df = pd.read_csv(csv_path)
    df["ts"] = pd.to_datetime(df["ts"])
    df = df.sort_values(["machine_id","ts"]).reset_index(drop=True)
    return df

def get_all_machine_ids(df_raw: pd.DataFrame):
    return sorted(df_raw["machine_id"].unique().tolist())

def build_initial_lag_state(df_raw: pd.DataFrame,
                            machine_id: str,
                            anchor_time: datetime | None = None,
                            n_lags: int = 3):
    df_m = df_raw[df_raw["machine_id"] == machine_id].copy()
    if df_m.empty:
        raise RuntimeError(f"no data for machine {machine_id}")

    if anchor_time is not None:
        df_m = df_m[df_m["ts"] <= anchor_time]

    df_m = df_m.sort_values("ts", ascending=False).reset_index(drop=True)

    if len(df_m) < 1:
        raise RuntimeError(f"machine {machine_id}: no rows before anchor_time")
    # fallback: kalau kurang dari n_lags, duplikasi nilai terakhir
    vals = df_m["value"].astype(float).tolist()
    while len(vals) < n_lags:
        vals.append(vals[-1])
    vals = vals[:n_lags]

    lag_state = {f"y_lag_{i+1}": float(v) for i,v in enumerate(vals)}
    anchor_used_ts = df_m.loc[0,"ts"]
    return lag_state, anchor_used_ts


def run_full_query(user_text, runner, df_raw, forecaster):
    """
    High level orchestrator:
    1. NLU (runner.predict)
    2. Interpret -> spec
    3. Jika SINGLE → prediksi satu mesin
       Jika ALL    → prediksi semua mesin yang ada di df_raw
    """

    # --- 1. classify + slot extract dari teks user
    runner_out = runner.predict(user_text)
    spec = interpret_query(user_text, runner_out)

    # --- 2. anchor time parsing
    # sekarang masih simple:
    # NOW_START -> pakai latest row
    # (kalau nanti ada ABSOLUTE_START kita parse natural datetime di sini)
    if spec["time"]["anchor_time_type"] == "NOW_START":
        anchor_time = None
    else:
        anchor_time = None  # TODO nanti parse spec["time"]["anchor_time_text"]

    # --- CASE A: scope == "SINGLE"
    if spec["scope"] == "SINGLE":
        target_machine = spec["machines"][0]

        init_state, anchor_used_ts = build_initial_lag_state(
            df_raw,
            machine_id=target_machine,
            anchor_time=anchor_time,
            n_lags=3,
        )

        exec_out = execute_spec(
            spec,
            forecaster=forecaster,
            context_state=init_state
        )

        exec_out["anchor_used_ts"]   = str(anchor_used_ts)
        exec_out["machine_context"]  = target_machine
        exec_out["scope_resolved"]   = "SINGLE"

        return {
            "runner_out": runner_out,
            "spec": spec,
            "exec_out": exec_out,
        }

    # --- CASE B: scope == "ALL"
    # Cara kita: loop setiap mesin real di df_raw, jalankan pipeline yang sama,
    # kumpulkan hasil dalam satu dict.
    elif spec["scope"] == "ALL":
        all_ids = get_all_machine_ids(df_raw)

        per_machine_result = {}
        for mid in all_ids:
            try:
                # 1. ambil lag actual utk mesin ini
                init_state, anchor_used_ts = build_initial_lag_state(
                    df_raw,
                    machine_id=mid,
                    anchor_time=anchor_time,
                    n_lags=3,
                )

                # 2. buat salinan spec khusus mesin ini
                #    kenapa? karena spec["machines"] sekarang ['ALL']
                #    tapi executor butuh nama mesin beneran untuk context kalau SINGLE-like
                #    kita bikin shallow copy aman
                spec_m = json.loads(json.dumps(spec))
                spec_m["machines"] = [mid]

                # NOTE:
                # kalau executor kamu butuh scope "ALL" buat range, biarin.
                # Tapi kalau executor kamu crash kalau scope != "SINGLE", 
                # kamu bisa force:
                # spec_m["scope"] = "SINGLE"
                # supaya code downstream treat sama seperti 1 mesin.
                #
                # Ini fleksibel tergantung versi execute_spec kamu sekarang.
                #
                # Rekomendasi awal: paksa jadi SINGLE.
                spec_m["scope"] = "SINGLE"

                # 3. run eksekusi forecast utk mesin ini
                machine_exec = execute_spec(
                    spec_m,
                    forecaster=forecaster,
                    context_state=init_state
                )

                # 4. lampirkan metadata anchor ts
                machine_exec["anchor_used_ts"]  = str(anchor_used_ts)
                machine_exec["machine_context"] = mid

                per_machine_result[mid] = machine_exec

            except Exception as e:
                # kalau satu mesin gagal (misal datanya terlalu sedikit),
                # kita tangkap supaya mesin lain tetap jalan
                per_machine_result[mid] = {
                    "error": str(e),
                    "machine_context": mid
                }

        final_exec_out = {
            "type":        "batch_forecast",
            "scope_resolved": "ALL",
            "machines":    all_ids,
            "per_machine": per_machine_result
        }

        return {
            "runner_out": runner_out,
            "spec": spec,
            "exec_out": final_exec_out,
        }

    # --- CASE C: MULTI (misal user sebut dua mesin spesifik)
    elif spec["scope"] == "MULTI":
        # spec["machines"] = ["PQ667","XP888A", ...]
        per_machine_result = {}
        for mid in spec["machines"]:
            try:
                init_state, anchor_used_ts = build_initial_lag_state(
                    df_raw,
                    machine_id=mid,
                    anchor_time=anchor_time,
                    n_lags=3,
                )

                spec_m = json.loads(json.dumps(spec))
                spec_m["machines"] = [mid]
                spec_m["scope"] = "SINGLE"

                machine_exec = execute_spec(
                    spec_m,
                    forecaster=forecaster,
                    context_state=init_state
                )

                machine_exec["anchor_used_ts"]  = str(anchor_used_ts)
                machine_exec["machine_context"] = mid

                per_machine_result[mid] = machine_exec

            except Exception as e:
                per_machine_result[mid] = {
                    "error": str(e),
                    "machine_context": mid
                }

        final_exec_out = {
            "type":        "batch_forecast",
            "scope_resolved": "MULTI",
            "machines":    spec["machines"],
            "per_machine": per_machine_result
        }

        return {
            "runner_out": runner_out,
            "spec": spec,
            "exec_out": final_exec_out,
        }

    else:
        # fallback sanity
        return {
            "runner_out": runner_out,
            "spec": spec,
            "exec_out": {
                "error": f"unhandled scope {spec['scope']}"
            }
        }


In [47]:
def get_all_machine_ids(df_raw: pd.DataFrame):
    return sorted(df_raw["machine_id"].unique().tolist())

def run_full_query(user_text,
                   runner,          # ForecastMultiRunner (NLP classifier)
                   df_raw,          # dataframe hasil load_raw_timeseries(...)
                   forecaster):     # HorizonForecaster / GRU forecaster

    # 1. classify + slot fill
    runner_out = runner.predict(user_text)
    spec = interpret_query(user_text, runner_out)

    # 2. tentukan mesin target utk state awal
    #    - scope SINGLE → pakai machines[0]
    #    - scope ALL → ??? (kita bisa pilih avg atau per-mesin loop)
    if spec["scope"] == "SINGLE":
        target_machine = spec["machines"][0]
    else:
        # sementara untuk ALL kita pilih salah satu mesin dulu / atau fail.
        # Nanti kita loop ALL satu per satu.
        target_machine = spec["machines"][0] if spec["machines"] else None

    # 3. tentukan anchor time
    #   - default: "NOW_START" → pakai timestamp terbaru (None in build fn)
    #   - kalau spec['time']['anchor_time_type'] == 'ABSOLUTE_START'
    #     dan ada spec['time']['anchor_time_text'] (misal "kemarin jam 10"),
    #     kamu nanti parse itu -> datetime
    if spec["time"]["anchor_time_type"] == "NOW_START":
        anchor_time = None
    else:
        # TODO: parse natural language time "kemarin jam 10"
        # sementara fallback None juga
        anchor_time = None

    # 4. build initial lag state dari CSV
    if target_machine is not None:
        init_state, anchor_used_ts = build_initial_lag_state(
            df_raw,
            machine_id=target_machine,
            anchor_time=anchor_time,
            n_lags=3
        )
    else:
        # kasus ALL → kita mungkin belum define, taruh None dulu
        init_state, anchor_used_ts = None, None

    # 5. panggil executor dengan init_state tersebut
    exec_out = execute_spec(
        spec,
        forecaster=forecaster,
        context_state=init_state
    )

    # 6. enrich exec_out dengan metadata waktu supaya jelas ke user
    exec_out["anchor_used_ts"] = str(anchor_used_ts) if anchor_used_ts is not None else None
    exec_out["machine_context"] = target_machine

    return {
        "runner_out": runner_out,
        "spec": spec,
        "exec_out": exec_out,
    }


In [50]:
############################################################
# 4. DEMO PIPELINE END-TO-END
############################################################

# 4a. init NLU multi-head
runner = ForecastMultiRunner(OUT_ROOT)

# 4b. init forecaster utk time-series numeric
# Ganti path ini dengan folder hasil training timeseries kamu
TS_RUN_ROOT = "/workspace/outputs_run_20251027_011402"  # <-- isi sesuai run_name kamu
forecaster = HorizonForecasterClassic(TS_RUN_ROOT)
# 4c. context_state realtime:
#   Harus diisi dengan lag aktual tiap mesin.
#   Di sini aku pakai dummy. Nanti di production:
#   - y_lag_1 = nilai sensor sekarang
#   - y_lag_2 = nilai sensor 1 step sebelumnya (misal 3 menit lalu)
#   - y_lag_3 = nilai sensor 2 step sebelumnya
context_state_example = {
    "y_lag_1": 105.2,
    "y_lag_2": 104.8,
    "y_lag_3": 104.5,
    # kalau modelmu pakai lag lebih banyak, tambahkan di sini
}

examples = [
    "nilai mesin XP888A 2 jam lagi berapa?",
    "coba prediksi semua mesin apakah akan ada anomali 2 jam kedepan?",
    "kapan XP888A bakal turun di bawah 70 derajat? kalau iya jam berapa pertama kalinya?",
    "andaikan data XP888A kemarin jam 10:00 gue ubah jadi 95, efeknya apa 2 jam lagi?",
    "ramalan performa semua mesin untuk 30 menit ke depan, interval 3 menit",
    "ramalan performa semua mesin untuk 48 menit, 24 menit, 12 menit ke depan",
    "ramalan performa semua mesin untuk 24 menit ke depan",
    "ramalan performa semua mesin untuk 12 menit ke depan",
    "ramalan performa semua mesin untuk 9 menit ke depan",
    "ramalan performa semua mesin untuk 6 menit ke depan",
    "ramalan performa semua mesin untuk 3 menit ke depan"
]
df_raw=pd.read_csv(r'/workspace/df_raw2.csv')
print("\n=== DEMO PREDIKSI ===")
for q in examples:
    out = run_full_query(q, runner, df_raw, forecaster)
    print("\nQ:", q)
    print(json.dumps(out, indent=2, ensure_ascii=False))
    print("-"*60)
    # print("\nQ:", q)

    # # (1) NLU heads
    # runner_out = runner.predict(q)

    # # (2) Normalized spec (fix horizon, scope, intent, etc)
    # spec = interpret_query(q, runner_out)

    # # (3) Time-series forecasting execution / rollout
    # exec_out = execute_spec(spec, forecaster, context_state_example)
    
    # # Print debug
    # # print("runner_out:")
    # # print(runner_out)
    # # print("\nspec:")
    # # print(spec)
    # # print("\nexec_out (final answer struct):")
    # # print(exec_out)
    # # print("-"*60)



=== DEMO PREDIKSI ===

Q: nilai mesin XP888A 2 jam lagi berapa?
{
  "runner_out": {
    "intent": {
      "label": "FORECAST_VALUE",
      "confidence": 0.9998
    },
    "scope": {
      "label": "SINGLE",
      "confidence": 0.9999
    },
    "time": {
      "time_granularity": "HOUR",
      "anchor_time_type": "NOW_START",
      "time_complexity": "SINGLE_POINT",
      "horizon_type": "FIXED_STEPS",
      "confidence": 0.9986
    },
    "horizon_minutes": {
      "y_start": 92.41140747070312,
      "y_end": 101.1116714477539
    },
    "threshold": {
      "metric": {
        "label": "NONE",
        "confidence": 0.9998
      },
      "operator": {
        "label": "NONE",
        "confidence": 0.9999
      },
      "value_predicted": {
        "y_val": -1.1409823894500732
      }
    },
    "what_if": {
      "flag": "NORMAL",
      "is_counterfactual": false,
      "confidence": 1.0
    },
    "route_hints": [
      "machine_entity_extractor",
      "forecast_numeric_pipeline",


In [1]:
# main.py
from Case3.config import OUT_ROOT, FORECASTER_ROOT, DATA_CSV
from Case3.nlu_runner import ForecastMultiRunner
from Case3.forecaster import HorizonForecasterClassic
from Case3.timeseries_state import load_raw_timeseries
from Case3.orchestrator import run_full_query
import json
def bootstrap():
    # 1. load model NLU (HF heads)
    runner = ForecastMultiRunner(root_dir=OUT_ROOT)

    # 2. load model klasik (linear+RF per horizon)
    forecaster = HorizonForecasterClassic(run_root=FORECASTER_ROOT)

    # 3. load data timeseries mentah
    df_raw = load_raw_timeseries(DATA_CSV)

    return runner, forecaster, df_raw

if __name__ == "__main__":
    runner, forecaster, df_raw = bootstrap()
    examples = [
        "Kira kira nilai mesin 2 jam kedepan berapa ya asumsi hari ini itu bulan februari?",
        # "coba prediksi semua mesin apakah akan ada anomali 2 jam kedepan?",
        # "kapan XP888A bakal turun di bawah 70 derajat? kalau iya jam berapa pertama kalinya?",
        # "andaikan data XP888A kemarin jam 10:00 gue ubah jadi 95, efeknya apa 2 jam lagi?",
        # "ramalan performa semua mesin untuk 30 menit ke depan, interval 3 menit",
        # "ramalan performa semua mesin untuk 48 menit, 24 menit, 12 menit ke depan",
        # "ramalan performa semua mesin untuk 24 menit ke depan",
        # "ramalan performa semua mesin untuk 12 menit ke depan",
        # "ramalan performa semua mesin untuk 9 menit ke depan",
        # "ramalan performa semua mesin untuk 6 menit ke depan",
        # "ramalan performa semua mesin untuk 3 menit ke depan"
    ]


    for q in examples:
        
        user_text = q
        result = run_full_query(
            user_text=user_text,
            runner=runner,
            df_raw=df_raw,
            forecaster=forecaster,
            n_lags=3,
        )
    
        
        print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "runner_out": {
    "intent": {
      "label": "WHAT_IF_FORECAST",
      "confidence": 0.647
    },
    "scope": {
      "label": "ALL",
      "confidence": 0.9816
    },
    "time": {
      "time_granularity": "HOUR",
      "anchor_time_type": "NOW_START",
      "time_complexity": "SINGLE_POINT",
      "horizon_type": "FIXED_STEPS",
      "confidence": 0.9984
    },
    "horizon_minutes": {
      "y_start": 92.20270538330078,
      "y_end": 101.35655212402344
    },
    "threshold": {
      "metric": {
        "label": "NONE",
        "confidence": 0.9998
      },
      "operator": {
        "label": "NONE",
        "confidence": 0.9999
      },
      "value_predicted": {
        "y_val": -1.1335124969482422
      }
    },
    "what_if": {
      "flag": "NORMAL",
      "is_counterfactual": false,
      "confidence": 0.9998
    },
    "route_hints": [
      "aggregate_all_machines",
      "counterfactual_simulation_pipeline",
      "time_parse_hour_single_point"
    ]
  },
  "spec"